In [ ]:
import torch

print("PyTorch versin:", torch.__version__)
print("Cuda available", torch.cuda.is_available())

if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import snapshot_download

dataset_path = snapshot_download(
    repo_id="ksanchez84/LUTSeg",
    repo_type="dataset",
    local_dir="data/LUTSeg"
)

print("Dataset tersimpan di:", dataset_path)

In [ ]:
import os

files = os.listdir(dataset_path)

for file in files:
  print(file)

In [ ]:
import json
with open("data/LUTSeg/class_map.json", "r") as f:
  class_map = json.load(f)

print(class_map)

In [ ]:
import os

image_files = os.listdir("data/LUTSeg/Images")
mask_files = os.listdir("data/LUTSeg/Masks")

print("Jumlah file gambar:", len(image_files))
print("Jumlah file mask:", len(mask_files))

print("\nContoh nama gambar:")
print(image_files[:5])

print("\nContoh nama mask:")
print(mask_files[:5])

In [ ]:

from pathlib import Path

image_dir = Path("data/LUTSeg/Images")
mask_dir = Path("data/LUTSeg/Masks")

image_files = sorted(image_dir.rglob("*.png"))
mask_files = sorted(mask_dir.rglob("*.png"))

print("Jumlah gambar:", len(image_files))
print("Jumlah mask:", len(mask_files))

print("\nContoh path gambar:")
print(image_files[:3])

print("\nContoh path mask:")
print(mask_files[:3])


In [ ]:

from collections import Counter

# Cari semua file di dalam folder Images
all_files = [
    f for f in image_dir.rglob("*")
    if f.is_file()
]

# Hitung jumlah file berdasarkan ekstensi
extensions = Counter(
    f.suffix.lower() for f in all_files
)

print("Format file:", extensions)
print("Total file:", len(all_files))

print("\nContoh file:")
for f in all_files[:5]:
    print(f)

In [ ]:

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Pilih satu gambar
image_path = all_files[0]

# Temukan mask yang sesuai
relative_path = image_path.relative_to(image_dir)
mask_path = mask_dir / relative_path.with_suffix(".png")

print("Gambar:", image_path)
print("Mask:", mask_path)
print("Mask ditemukan:", mask_path.exists())

# Baca gambar dan mask
image = Image.open(image_path).convert("RGB")
mask = np.array(Image.open(mask_path))

print("Ukuran gambar:", image.size)
print("Ukuran mask:", mask.shape)
print("Label unik:", np.unique(mask))


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Foto luka asli
axes[0].imshow(image)
axes[0].set_title("Original Wound Image")
axes[0].axis("off")

# Segmentation mask
mask_display = np.ma.masked_equal(mask, 255)

axes[1].imshow(
    mask_display,
    cmap="tab10",
    vmin=0,
    vmax=5,
    interpolation="nearest"
)
axes[1].set_title("Segmentation Mask")
axes[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import json
import numpy as np

# Baca informasi kelas
with open("data/LUTSeg/class_map.json", "r") as f:
    class_map = json.load(f)

print("Daftar kelas:")
print(class_map)

# Hitung jumlah piksel setiap label
labels, counts = np.unique(mask, return_counts=True)

print("\nDistribusi piksel:")
for label, count in zip(labels, counts):
    print(f"Label {label}: {count:,} piksel")


In [ ]:

# Ubah gambar menjadi array NumPy
image_array = np.array(image)

# Buat salinan gambar untuk overlay
overlay = image_array.copy()

# Tentukan warna tiap jaringan (RGB)
colors = {
    1: [255, 105, 180],  # Epithelial - Pink
    2: [255, 215, 0],    # Slough - Yellow
    3: [255, 60, 60],    # Granulation - Red
    4: [40, 40, 40],     # Necrotic - Dark
    5: [40, 160, 255]    # Other - Blue
}

# Transparansi warna
alpha = 0.5

# Warnai setiap kelas jaringan
for class_id, color in colors.items():
    region = mask == class_id

    overlay[region] = (
        (1 - alpha) * image_array[region]
        + alpha * np.array(color)
    ).astype(np.uint8)


In [ ]:

plt.figure(figsize=(8, 8))

plt.imshow(overlay)
plt.title("WoundVision AI - Ground Truth Overlay")
plt.axis("off")

plt.show()


In [ ]:

from collections import Counter

# Ambil seluruh file mask
mask_files = sorted(mask_dir.rglob("*.png"))

# Siapkan penghitung
pixel_counts = Counter()
image_counts = Counter()

# Baca seluruh mask satu per satu
for mask_path in mask_files:
    mask_array = np.array(Image.open(mask_path))

    labels, counts = np.unique(
        mask_array,
        return_counts=True
    )

    for label, count in zip(labels, counts):
        pixel_counts[int(label)] += int(count)
        image_counts[int(label)] += 1

print("Total mask:", len(mask_files))
print("Jumlah piksel per kelas:", pixel_counts)


In [ ]:

# Ambil nama kelas dari class_map
class_names = {
    value: name
    for name, value in class_map.items()
}

# Total piksel jaringan, tanpa background dan ignore
total_tissue = sum(
    pixel_counts[i] for i in range(1, 6)
)

print("DISTRIBUSI JARINGAN LUTSeg")
print("-" * 45)

for class_id in range(1, 6):
    count = pixel_counts[class_id]
    percentage = 100 * count / total_tissue

    print(
        f"{class_names[class_id]}: "
        f"{percentage:.2f}% "
        f"({image_counts[class_id]} gambar)"
    )

print("-" * 45)
print("Background:", pixel_counts[0])
print("Ignore:", pixel_counts[255])


In [ ]:

# Ambil ID jaringan selain background
tissue_ids = [1, 2, 3, 4, 5]

# Ambil nama jaringan
names = [class_names[i] for i in tissue_ids]

# Hitung persentase setiap jaringan
percentages = [
    100 * pixel_counts[i] / total_tissue
    for i in tissue_ids
]

# Membuat grafik batang
plt.figure(figsize=(10, 5))

plt.bar(names, percentages)

plt.title("LUTSeg - Tissue Class Distribution")
plt.xlabel("Tissue Class")
plt.ylabel("Percentage (%)")

plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:

from pathlib import Path

data_dir = Path("data/LUTSeg")

# Baca daftar training
with open(data_dir / "train.txt", "r") as f:
    train_files = f.read().splitlines()

# Baca daftar validation
with open(data_dir / "val.txt", "r") as f:
    val_files = f.read().splitlines()

print("Jumlah training:", len(train_files))
print("Jumlah validation:", len(val_files))

print("\nContoh training:")
print(train_files[:5])

print("\nContoh validation:")
print(val_files[:5])


In [ ]:

# Ubah list menjadi set
train_set = set(train_files)
val_set = set(val_files)

# Cari entri yang muncul di kedua split
overlap = train_set.intersection(val_set)

print("Duplikat entri:", len(overlap))
print("Contoh overlap:", list(overlap)[:5])


In [ ]:

# Fungsi untuk mengambil ID pasien
def get_patient_id(entry):
    image_path, mask_path = entry.split()
    patient_id = Path(image_path).parts[1]
    return patient_id

# Ambil pasien unik dari masing-masing split
train_patients = set(
    get_patient_id(entry) for entry in train_files
)

val_patients = set(
    get_patient_id(entry) for entry in val_files
)

# Periksa apakah ada pasien yang sama
patient_overlap = train_patients.intersection(val_patients)

print("Jumlah pasien training:", len(train_patients))
print("Jumlah pasien validation:", len(val_patients))
print("Patient overlap:", patient_overlap)

print("\nTotal pasien unik:",
      len(train_patients | val_patients))


In [ ]:

import torch
import numpy as np

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset

class WoundDataset(Dataset):

    def __init__(self, root_dir, entries):
        self.root_dir = Path(root_dir)
        self.entries = entries

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, index):

        # Ambil path foto dan mask
        entry = self.entries[index]
        image_path, mask_path = entry.split()

        # Buka gambar
        image = Image.open(
            self.root_dir / image_path
        ).convert("RGB")

        mask = Image.open(
            self.root_dir / mask_path
        )

        # Resize foto dan mask
        image = image.resize(
            (320, 320),
            Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (320, 320),
            Image.Resampling.NEAREST
        )

        # Konversi ke NumPy
        image = np.array(image, dtype=np.float32) / 255.0
        mask = np.array(mask, dtype=np.int64)

        # Konversi ke tensor
        image = torch.from_numpy(image).permute(2, 0, 1)
        mask = torch.from_numpy(mask).long()

        return image, mask


In [ ]:

# Buat dataset training dan validation
train_dataset = WoundDataset(data_dir, train_files)
val_dataset = WoundDataset(data_dir, val_files)

# Ambil satu sampel
image_tensor, mask_tensor = train_dataset[0]

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

print("Image shape:", image_tensor.shape)
print("Mask shape:", mask_tensor.shape)

print("Image dtype:", image_tensor.dtype)
print("Mask dtype:", mask_tensor.dtype)

print("Unique labels:", torch.unique(mask_tensor))


In [ ]:

from torch.utils.data import DataLoader

# Tentukan ukuran batch
BATCH_SIZE = 4

# DataLoader untuk training
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# DataLoader untuk validation
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


In [ ]:

# Ambil satu batch dari training
images, masks = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Mask batch shape:", masks.shape)

print("Image dtype:", images.dtype)
print("Mask dtype:", masks.dtype)

print("Unique labels:", torch.unique(masks))


In [ ]:

from collections import Counter

class_presence = Counter()
lost_classes = Counter()

for entry in train_files + val_files:

    # Ambil lokasi mask
    _, mask_path = entry.split()

    # Baca mask asli
    original = np.array(
        Image.open(data_dir / mask_path)
    )

    # Resize menjadi 320 x 320
    resized = np.array(
        Image.fromarray(original).resize(
            (320, 320),
            Image.Resampling.NEAREST
        )
    )

    # Bandingkan label sebelum dan sesudah
    original_labels = set(np.unique(original))
    resized_labels = set(np.unique(resized))

    for class_id in range(1, 6):

        if class_id in resized_labels:
            class_presence[class_id] += 1

        if class_id in original_labels and class_id not in resized_labels:
            lost_classes[class_id] += 1

print("Jumlah gambar per kelas setelah resize:")
for class_id in range(1, 6):
    print(class_names[class_id], ":", class_presence[class_id])

print("\nKelas yang hilang setelah resize:")
print(lost_classes)


In [ ]:
!pip install -q segmentation-models-pytorch


In [ ]:

import segmentation_models_pytorch as smp
import torch

# Tentukan perangkat komputasi
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Jumlah kelas: 5 jaringan + background
NUM_CLASSES = 6

# Bangun model U-Net
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None
)

# Pindahkan model ke GPU
model = model.to(device)

print("Device:", device)
print("Model:", model.__class__.__name__)


In [ ]:

# Ambil satu batch
images, masks = next(iter(train_loader))

# Pindahkan gambar ke GPU
images = images.to(device)

# Aktifkan mode evaluasi untuk pengujian
model.eval()

# Jalankan forward pass tanpa menghitung gradient
with torch.no_grad():
    outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Device:", outputs.device)


In [ ]:

def count_classes(entries):
    counts = {i: 0 for i in range(1, 6)}

    for entry in entries:
        image_path, mask_path = entry.split()

        mask = np.array(
            Image.open(data_dir / mask_path)
        )

        unique_labels = np.unique(mask)

        for class_id in range(1, 6):
            if class_id in unique_labels:
                counts[class_id] += 1

    return counts


# Hitung masing-masing split
train_counts = count_classes(train_files)
val_counts = count_classes(val_files)

# Tampilkan hasil
print("CLASS DISTRIBUTION BY SPLIT")
print("-" * 50)

for class_id in range(1, 6):
    name = class_names[class_id]

    print(
        f"{name}: "
        f"Train={train_counts[class_id]}, "
        f"Val={val_counts[class_id]}"
    )


In [ ]:

# Statistik ImageNet
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

def normalize_images(images):

    mean = torch.tensor(MEAN).view(1, 3, 1, 1)
    std = torch.tensor(STD).view(1, 3, 1, 1)

    mean = mean.to(images.device)
    std = std.to(images.device)

    normalized = (images - mean) / std

    return normalized


In [ ]:

# Ambil satu batch
images, masks = next(iter(train_loader))

# Normalisasi
normalized_images = normalize_images(images)

print("Shape:", normalized_images.shape)

print("Sebelum normalisasi:")
print("Min:", images.min().item())
print("Max:", images.max().item())

print("\nSesudah normalisasi:")
print("Min:", normalized_images.min().item())
print("Max:", normalized_images.max().item())

print("\nMask labels:", torch.unique(masks))


In [ ]:

import torch.nn as nn
import segmentation_models_pytorch as smp

# Cross Entropy Loss
ce_loss = nn.CrossEntropyLoss(
    ignore_index=255
)

# Dice Loss
dice_loss = smp.losses.DiceLoss(
    mode="multiclass",
    from_logits=True,
    ignore_index=255
)

# Gabungkan kedua loss
def combined_loss(outputs, targets):

    loss_ce = ce_loss(outputs, targets)
    loss_dice = dice_loss(outputs, targets)

    total_loss = loss_ce + loss_dice

    return total_loss


In [ ]:

import torch.optim as optim

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.0001
)

print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])


In [ ]:

# Ambil satu mini-batch
images, masks = next(iter(train_loader))

# Normalisasi dan pindahkan ke GPU
images = normalize_images(images).to(device)
masks = masks.to(device)

# Uji prediksi tanpa training
model.eval()

with torch.no_grad():

    outputs = model(images)

    loss = combined_loss(outputs, masks)

print("Output shape:", outputs.shape)
print("Loss:", loss.item())
print("Loss valid:", torch.isfinite(loss).item())


In [ ]:

# Aktifkan mode training
model.train()

total_loss = 0.0
total_samples = 0

# Iterasi setiap mini-batch
for batch_idx, (images, masks) in enumerate(train_loader):

    # Normalisasi dan pindahkan ke GPU
    images = normalize_images(images).to(device)
    masks = masks.to(device)

    # Bersihkan gradient sebelumnya
    optimizer.zero_grad(set_to_none=True)

    # Forward propagation
    outputs = model(images)

    # Hitung loss
    loss = combined_loss(outputs, masks)

    # Backpropagation
    loss.backward()

    # Perbarui bobot model
    optimizer.step()

    # Akumulasi loss
    batch_size = images.size(0)
    total_loss += loss.item() * batch_size
    total_samples += batch_size

    # Tampilkan progress setiap 5 batch
    if (batch_idx + 1) % 5 == 0:
        print(
            f"Batch {batch_idx + 1}/{len(train_loader)} "
            f"- Loss: {loss.item():.4f}"
        )

# Hitung rata-rata loss satu epoch
avg_train_loss = total_loss / total_samples

print("\nEpoch 1 selesai!")
print(f"Average Training Loss: {avg_train_loss:.4f}")


In [ ]:

# Aktifkan mode evaluasi
model.eval()

val_total_loss = 0.0
val_total_samples = 0

# Tidak menghitung gradient
with torch.no_grad():

    for images, masks in val_loader:

        # Normalisasi gambar
        images = normalize_images(images)

        # Pindahkan data ke GPU
        images = images.to(device)
        masks = masks.to(device)

        # Forward pass
        outputs = model(images)

        # Hitung validation loss
        loss = combined_loss(outputs, masks)

        # Akumulasi loss
        batch_size = images.size(0)

        val_total_loss += loss.item() * batch_size
        val_total_samples += batch_size

# Rata-rata validation loss
avg_val_loss = val_total_loss / val_total_samples

print("VALIDATION RESULT")
print("-------------------------")
print(f"Training Loss: {avg_train_loss:.4f}")
print(f"Validation Loss: {avg_val_loss:.4f}")


In [ ]:

NUM_CLASSES = 6

# Confusion matrix untuk 6 kelas
confusion = torch.zeros(
    (NUM_CLASSES, NUM_CLASSES),
    dtype=torch.int64
)

model.eval()

with torch.no_grad():

    for images, masks in val_loader:

        images = normalize_images(images).to(device)
        masks = masks.to(device)

        # Prediksi model
        outputs = model(images)

        # Ambil kelas dengan skor tertinggi
        predictions = torch.argmax(outputs, dim=1)

        # Abaikan piksel ignore
        valid = (masks >= 0) & (masks < NUM_CLASSES)

        targets = masks[valid]
        preds = predictions[valid]

        # Hitung confusion matrix
        indices = targets * NUM_CLASSES + preds

        counts = torch.bincount(
            indices,
            minlength=NUM_CLASSES ** 2
        )

        confusion += counts.reshape(
            NUM_CLASSES, NUM_CLASSES
        ).cpu()

print("Confusion Matrix:")
print(confusion)


In [ ]:

print("VALIDATION METRICS - EPOCH 1")
print("-" * 65)

dice_scores = []
iou_scores = []

for class_id in range(NUM_CLASSES):

    # True Positive
    tp = confusion[class_id, class_id].item()

    # False Positive
    fp = confusion[:, class_id].sum().item() - tp

    # False Negative
    fn = confusion[class_id, :].sum().item() - tp

    # Hitung Dice dan IoU
    dice_denominator = 2 * tp + fp + fn
    iou_denominator = tp + fp + fn

    if dice_denominator == 0:
        print(f"{class_names[class_id]}: N/A")
        continue

    dice = (2 * tp) / dice_denominator
    iou = tp / iou_denominator

    print(
        f"{class_names[class_id]}: "
        f"Dice={dice:.4f}, IoU={iou:.4f}"
    )

    # Macro-average tanpa background
    if class_id != 0:
        dice_scores.append(dice)
        iou_scores.append(iou)

print("-" * 65)

print(
    f"Mean Tissue Dice: "
    f"{np.mean(dice_scores):.4f}"
)

print(
    f"Mean Tissue IoU: "
    f"{np.mean(iou_scores):.4f}"
)


In [ ]:

from matplotlib.colors import ListedColormap

# Warna untuk setiap kelas
colors = [
    "#111827",  # Background
    "#FF69B4",  # Epithelial
    "#FFD700",  # Slough
    "#FF3C3C",  # Granulation
    "#696969",  # Necrotic
    "#28A0FF"   # Other
]

cmap = ListedColormap(colors)

# Ambil satu batch validation
val_images, val_masks = next(iter(val_loader))

# Prediksi menggunakan GPU
model.eval()

with torch.no_grad():
    inputs = normalize_images(val_images).to(device)
    outputs = model(inputs)
    predictions = outputs.argmax(dim=1).cpu()

# Pilih gambar pertama
image = val_images[0].permute(1, 2, 0).numpy()
ground_truth = val_masks[0].numpy()
prediction = predictions[0].numpy()

# Tampilkan hasil
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image)
axes[0].set_title("Original Image")

axes[1].imshow(
    ground_truth, cmap=cmap, vmin=0, vmax=5
)
axes[1].set_title("Ground Truth")

axes[2].imshow(
    prediction, cmap=cmap, vmin=0, vmax=5
)
axes[2].set_title("U-Net Prediction")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

save_dir = Path("/content/drive/MyDrive/WoundVisionAI")
save_dir.mkdir(parents=True, exist_ok=True)

# Simpan model setelah epoch 1
torch.save({
    "epoch": 1,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "train_loss": avg_train_loss,
    "val_loss": avg_val_loss
}, save_dir / "epoch_01.pth")

print("Checkpoint epoch 1 berhasil disimpan!")


In [ ]:

history = {
    "epoch": [1],
    "train_loss": [avg_train_loss],
    "val_loss": [avg_val_loss]
}

best_val_loss = avg_val_loss

for epoch in range(2, 7):

    # TRAINING
    model.train()
    train_total = 0

    for images, masks in train_loader:

        images = normalize_images(images).to(device)
        masks = masks.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = combined_loss(outputs, masks)

        loss.backward()
        optimizer.step()

        train_total += loss.item() * images.size(0)

    train_loss = train_total / len(train_dataset)

    # VALIDATION
    model.eval()
    val_total = 0

    with torch.no_grad():
        for images, masks in val_loader:

            images = normalize_images(images).to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = combined_loss(outputs, masks)

            val_total += loss.item() * images.size(0)

    val_loss = val_total / len(val_dataset)

    # Simpan riwayat
    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # Simpan model terbaik
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss": val_loss
        }, save_dir / "best_model.pth")

    print(
        f"Epoch {epoch}/6 | "
        f"Train: {train_loss:.4f} | "
        f"Val: {val_loss:.4f}"
    )


In [ ]:

checkpoint = torch.load(
    save_dir / "best_model.pth",
    map_location="cpu",
    weights_only=True
)

print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["val_loss"])


In [ ]:

import torch
import numpy as np

NUM_CLASSES = 6

# Siapkan confusion matrix kosong
confusion = torch.zeros(
    (NUM_CLASSES, NUM_CLASSES),
    dtype=torch.int64
)

# Aktifkan mode evaluasi
model.eval()

with torch.no_grad():

    for images, masks in val_loader:

        # Normalisasi dan pindahkan ke GPU
        images = normalize_images(images).to(device)
        masks = masks.to(device)

        # Prediksi menggunakan U-Net
        outputs = model(images)

        # Ambil kelas dengan skor tertinggi
        predictions = torch.argmax(outputs, dim=1)

        # Abaikan label ignore (255)
        valid = (masks >= 0) & (masks < NUM_CLASSES)

        targets = masks[valid]
        preds = predictions[valid]

        # Hitung confusion matrix
        indices = targets * NUM_CLASSES + preds

        counts = torch.bincount(
            indices,
            minlength=NUM_CLASSES ** 2
        )

        confusion += counts.reshape(
            NUM_CLASSES, NUM_CLASSES
        ).cpu()

print("CONFUSION MATRIX - EPOCH 6")
print(confusion)


In [ ]:

# Nama kelas LUTSeg
class_names = {
    0: "Background",
    1: "Epithelial tissue",
    2: "Slough",
    3: "Granulation tissue",
    4: "Necrotic tissue",
    5: "Other"
}

dice_scores = []
iou_scores = []

print("VALIDATION METRICS - EPOCH 6")
print("-" * 65)

for class_id in range(NUM_CLASSES):

    # True Positive
    tp = confusion[class_id, class_id].item()

    # False Positive
    fp = confusion[:, class_id].sum().item() - tp

    # False Negative
    fn = confusion[class_id, :].sum().item() - tp

    # Hitung Dice dan IoU
    dice_denominator = 2 * tp + fp + fn
    iou_denominator = tp + fp + fn

    if dice_denominator == 0:
        print(f"{class_names[class_id]}: N/A")
        continue

    dice = 2 * tp / dice_denominator
    iou = tp / iou_denominator

    print(
        f"{class_names[class_id]}: "
        f"Dice={dice:.4f}, IoU={iou:.4f}"
    )

    # Rata-rata tanpa background
    if class_id != 0:
        dice_scores.append(dice)
        iou_scores.append(iou)

print("-" * 65)

print(f"Mean Tissue Dice: {np.mean(dice_scores):.4f}")
print(f"Mean Tissue IoU: {np.mean(iou_scores):.4f}")


In [ ]:

# Siapkan penghitung setiap kelas
train_pixel_counts = torch.zeros(
    NUM_CLASSES,
    dtype=torch.int64
)

# Hitung piksel dari seluruh training dataset
for image, mask in train_dataset:

    # Abaikan label di luar kelas 0-5
    valid_mask = mask[
        (mask >= 0) & (mask < NUM_CLASSES)
    ]

    # Hitung jumlah piksel tiap kelas
    counts = torch.bincount(
        valid_mask.flatten(),
        minlength=NUM_CLASSES
    )

    train_pixel_counts += counts

# Hitung total piksel
total_pixels = train_pixel_counts.sum().item()

# Hitung total piksel jaringan tanpa background
total_tissue_pixels = train_pixel_counts[1:].sum().item()

print("TRAINING PIXEL DISTRIBUTION")
print("-" * 55)

for class_id in range(NUM_CLASSES):

    count = train_pixel_counts[class_id].item()

    percentage = count / total_pixels * 100

    print(
        f"{class_names[class_id]}: "
        f"{count:,} pixels ({percentage:.4f}%)"
    )

print("-" * 55)
print("Total tissue pixels:", total_tissue_pixels)


In [ ]:

# Ambil jumlah piksel training
counts = train_pixel_counts.float()

# Gunakan kelas jaringan umum sebagai referensi
reference = counts[1:4].max()

# Hitung bobot dengan inverse square root
class_weights = torch.sqrt(
    reference / counts.clamp(min=1)
)

# Batasi bobot maksimum
class_weights = torch.clamp(
    class_weights,
    max=3.0
)

# Kurangi dominasi background
class_weights[0] = 0.25

# Other terlalu sedikit untuk dibobot tinggi
class_weights[5] = 1.0

# Tampilkan bobot
print("CLASS WEIGHTS")
print("-" * 40)

for class_id in range(NUM_CLASSES):
    print(
        f"{class_names[class_id]}: "
        f"{class_weights[class_id]:.4f}"
    )


In [ ]:

import torch.nn as nn

# Weighted Cross Entropy
weighted_ce_loss = nn.CrossEntropyLoss(
    weight=class_weights.to(device),
    ignore_index=255
)

# Kombinasikan dengan Dice Loss
def combined_loss_v2(outputs, targets):

    loss_ce = weighted_ce_loss(outputs, targets)
    loss_dice = dice_loss(outputs, targets)

    total_loss = loss_ce + loss_dice

    return total_loss


In [ ]:

import shutil

# Lokasi checkpoint terbaik epoch 6
source = save_dir / "best_model.pth"

# Buat salinan untuk eksperimen
baseline_path = save_dir / "baseline_epoch06.pth"

shutil.copy2(source, baseline_path)

# Periksa checkpoint
checkpoint = torch.load(
    baseline_path,
    map_location=device,
    weights_only=True
)

print("Baseline epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])
print("Backup exists:", baseline_path.exists())


In [ ]:

# Muat kembali bobot epoch 6
model.load_state_dict(
    checkpoint["model_state"]
)

# Muat kembali status optimizer
optimizer.load_state_dict(
    checkpoint["optimizer_state"]
)

# Verifikasi
print("Model restored from epoch:", checkpoint["epoch"])

# Pastikan weighted loss sudah tersedia
print("V2 Class Weights:")
print(class_weights)

print("Ready for V2 training!")


In [ ]:

# Mode training
model.train()

total_train_loss = 0.0
total_samples = 0

for images, masks in train_loader:

    # Preprocessing
    images = normalize_images(images).to(device)
    masks = masks.to(device)

    # Bersihkan gradient
    optimizer.zero_grad(set_to_none=True)

    # Forward propagation
    outputs = model(images)

    # Gunakan weighted loss V2
    loss = combined_loss_v2(outputs, masks)

    # Backpropagation
    loss.backward()

    # Update bobot
    optimizer.step()

    # Akumulasi loss
    batch_size = images.size(0)
    total_train_loss += loss.item() * batch_size
    total_samples += batch_size

avg_train_loss_v2 = total_train_loss / total_samples

print("V2 - EPOCH 7")
print(f"Training Loss: {avg_train_loss_v2:.4f}")


In [ ]:

torch.save({
    "epoch": 7,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "train_loss": avg_train_loss_v2,
    "class_weights": class_weights.cpu()
}, save_dir / "v2_epoch07.pth")

print("V2 Epoch 7 checkpoint saved!")


In [ ]:

# Siapkan confusion matrix baru
confusion_v2 = torch.zeros(
    (NUM_CLASSES, NUM_CLASSES),
    dtype=torch.int64
)

# Mode evaluasi
model.eval()

with torch.no_grad():

    for images, masks in val_loader:

        images = normalize_images(images).to(device)
        masks = masks.to(device)

        # Prediksi V2
        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        # Ambil piksel valid
        valid = (masks >= 0) & (masks < NUM_CLASSES)

        targets = masks[valid]
        preds = predictions[valid]

        # Hitung confusion matrix
        indices = targets * NUM_CLASSES + preds

        counts = torch.bincount(
            indices,
            minlength=NUM_CLASSES ** 2
        )

        confusion_v2 += counts.reshape(
            NUM_CLASSES, NUM_CLASSES
        ).cpu()

print("CONFUSION MATRIX - V2 EPOCH 7")
print(confusion_v2)


In [ ]:

dice_v2 = []
iou_v2 = []

print("VALIDATION METRICS - V2 EPOCH 7")
print("-" * 60)

for class_id in range(NUM_CLASSES):

    tp = confusion_v2[class_id, class_id].item()
    fp = confusion_v2[:, class_id].sum().item() - tp
    fn = confusion_v2[class_id, :].sum().item() - tp

    denominator = 2 * tp + fp + fn

    if denominator == 0:
        print(f"{class_names[class_id]}: N/A")
        continue

    dice = (2 * tp) / denominator
    iou = tp / (tp + fp + fn)

    print(
        f"{class_names[class_id]}: "
        f"Dice={dice:.4f}, IoU={iou:.4f}"
    )

    if class_id != 0:
        dice_v2.append(dice)
        iou_v2.append(iou)

print("-" * 60)
print(f"Mean Tissue Dice: {np.mean(dice_v2):.4f}")
print(f"Mean Tissue IoU: {np.mean(iou_v2):.4f}")


In [ ]:

# Lokasi checkpoint
baseline_path = save_dir / "baseline_epoch06.pth"
v2_path = save_dir / "v2_epoch07.pth"

# Pastikan keduanya tersimpan
print("Baseline saved:", baseline_path.exists())
print("V2 saved:", v2_path.exists())

assert baseline_path.exists()
assert v2_path.exists()

# Baca checkpoint baseline
checkpoint = torch.load(
    baseline_path,
    map_location=device,
    weights_only=True
)

# Kembalikan model dan optimizer ke epoch 6
model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])

print("Restored epoch:", checkpoint["epoch"])
print("Ready for V1 control!")


In [ ]:

# Aktifkan mode training
model.train()

total_loss_v1 = 0.0
total_samples_v1 = 0

for images, masks in train_loader:

    # Normalisasi dan pindahkan ke GPU
    images = normalize_images(images).to(device)
    masks = masks.to(device)

    # Bersihkan gradient
    optimizer.zero_grad(set_to_none=True)

    # Forward pass
    outputs = model(images)

    # Gunakan loss ORIGINAL (V1)
    loss = combined_loss(outputs, masks)

    # Backpropagation
    loss.backward()
    optimizer.step()

    # Hitung total loss
    batch_size = images.size(0)

    total_loss_v1 += loss.item() * batch_size
    total_samples_v1 += batch_size

avg_loss_v1 = total_loss_v1 / total_samples_v1

print("V1 CONTROL - EPOCH 7")
print(f"Training Loss: {avg_loss_v1:.4f}")


In [ ]:

torch.save({
    "epoch": 7,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "train_loss": avg_loss_v1
}, save_dir / "v1_epoch07.pth")

print("V1 Epoch 7 saved!")


In [ ]:

# Siapkan confusion matrix
confusion_v1 = torch.zeros(
    (NUM_CLASSES, NUM_CLASSES),
    dtype=torch.int64
)

model.eval()

with torch.no_grad():
    for images, masks in val_loader:

        # Preprocessing
        images = normalize_images(images).to(device)
        masks = masks.to(device)

        # Prediksi model
        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        # Ambil piksel valid
        valid = (masks >= 0) & (masks < NUM_CLASSES)

        targets = masks[valid]
        preds = predictions[valid]

        # Hitung pasangan kelas
        indices = targets * NUM_CLASSES + preds

        counts = torch.bincount(
            indices,
            minlength=NUM_CLASSES ** 2
        )

        confusion_v1 += counts.reshape(
            NUM_CLASSES, NUM_CLASSES
        ).cpu()

print("CONFUSION MATRIX - V1 EPOCH 7")
print(confusion_v1)


In [ ]:

dice_v1 = []
iou_v1 = []

print("VALIDATION METRICS - V1 EPOCH 7")
print("-" * 60)

for class_id in range(NUM_CLASSES):

    tp = confusion_v1[class_id, class_id].item()

    fp = confusion_v1[:, class_id].sum().item() - tp
    fn = confusion_v1[class_id, :].sum().item() - tp

    dice_denominator = 2 * tp + fp + fn
    iou_denominator = tp + fp + fn

    if dice_denominator == 0:
        print(f"{class_names[class_id]}: N/A")
        continue

    dice = (2 * tp) / dice_denominator
    iou = tp / iou_denominator

    print(
        f"{class_names[class_id]}: "
        f"Dice={dice:.4f}, IoU={iou:.4f}"
    )

    if class_id != 0:
        dice_v1.append(dice)
        iou_v1.append(iou)

print("-" * 60)
print(f"Mean Tissue Dice: {np.mean(dice_v1):.4f}")
print(f"Mean Tissue IoU: {np.mean(iou_v1):.4f}")


In [ ]:

from torch.utils.data import WeightedRandomSampler

# Tandai gambar yang mengandung Necrotic
necrotic_presence = []

for entry in train_files:

    image_path, mask_path = entry.split()

    mask = np.array(
        Image.open(data_dir / mask_path)
    )

    has_necrotic = np.any(mask == 4)

    necrotic_presence.append(bool(has_necrotic))

# Berikan bobot lebih besar pada gambar Necrotic
sample_weights = [
    3.0 if has_necrotic else 1.0
    for has_necrotic in necrotic_presence
]

# Buat sampler
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True,
    generator=torch.Generator().manual_seed(42)
)

# Uji sampel yang dipilih
sampled_indices = list(iter(sampler))

sampled_necrotic = sum(
    necrotic_presence[i]
    for i in sampled_indices
)

print("Original Necrotic Images:", sum(necrotic_presence))
print("Total Training Images:", len(train_dataset))
print("Sampled Necrotic Images:", sampled_necrotic)
print("Total Sampled:", len(sampled_indices))


In [ ]:

from torch.utils.data import DataLoader

train_loader_v3 = DataLoader(
    train_dataset,
    batch_size=4,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

print("Total batch V3:", len(train_loader_v3))


In [ ]:

# Ambil satu batch dari DataLoader V3
images_v3, masks_v3 = next(iter(train_loader_v3))

print("Image shape:", images_v3.shape)
print("Mask shape:", masks_v3.shape)

# Periksa gambar yang mengandung Necrotic
has_necrotic = (masks_v3 == 4).flatten(1).any(dim=1)

print("Necrotic per image:", has_necrotic)
print("Necrotic images in batch:", has_necrotic.sum().item())


In [ ]:

# Baca checkpoint V2
checkpoint_v2 = torch.load(
    save_dir / "v2_epoch07.pth",
    map_location=device,
    weights_only=True
)

# Kembalikan model
model.load_state_dict(
    checkpoint_v2["model_state"]
)

# Kembalikan optimizer
optimizer.load_state_dict(
    checkpoint_v2["optimizer_state"]
)

# Atur seed eksperimen
torch.manual_seed(2026)
torch.cuda.manual_seed_all(2026)

# Reset generator sampler
sampler.generator.manual_seed(42)

print("Starting epoch:", checkpoint_v2["epoch"])
print("Ready for V3 training!")


In [ ]:

# Aktifkan mode training
model.train()

total_loss_v3 = 0.0
total_samples_v3 = 0

# Gunakan DataLoader V3
for images, masks in train_loader_v3:

    # Normalisasi input
    images = normalize_images(images).to(device)
    masks = masks.to(device)

    # Bersihkan gradient
    optimizer.zero_grad(set_to_none=True)

    # Forward pass
    outputs = model(images)

    # Weighted Loss V2
    loss = combined_loss_v2(outputs, masks)

    # Backpropagation
    loss.backward()

    # Update parameter
    optimizer.step()

    # Akumulasi loss
    batch_size = images.size(0)

    total_loss_v3 += loss.item() * batch_size
    total_samples_v3 += batch_size

# Rata-rata training loss
avg_loss_v3 = total_loss_v3 / total_samples_v3

print("V3 - EPOCH 8")
print(f"Training Loss: {avg_loss_v3:.4f}")
print(f"Samples processed: {total_samples_v3}")


In [ ]:

torch.save({
    "epoch": 8,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "train_loss": avg_loss_v3,
    "sampling": "necrotic_aware"
}, save_dir / "v3_epoch08.pth")

print("V3 checkpoint saved!")


In [ ]:

# Buat confusion matrix baru
confusion_v3 = torch.zeros(
    (NUM_CLASSES, NUM_CLASSES),
    dtype=torch.int64
)

model.eval()

with torch.no_grad():

    for images, masks in val_loader:

        # Normalisasi dan pindahkan ke GPU
        images = normalize_images(images).to(device)
        masks = masks.to(device)

        # Prediksi model V3
        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        # Abaikan piksel ignore
        valid = (masks >= 0) & (masks < NUM_CLASSES)

        targets = masks[valid]
        preds = predictions[valid]

        # Hitung confusion matrix
        indices = targets * NUM_CLASSES + preds

        counts = torch.bincount(
            indices,
            minlength=NUM_CLASSES ** 2
        )

        confusion_v3 += counts.reshape(
            NUM_CLASSES, NUM_CLASSES
        ).cpu()

print("CONFUSION MATRIX - V3 EPOCH 8")
print(confusion_v3)


In [ ]:

dice_v3 = []
iou_v3 = []

print("VALIDATION METRICS - V3 EPOCH 8")
print("-" * 60)

for class_id in range(NUM_CLASSES):

    tp = confusion_v3[class_id, class_id].item()
    fp = confusion_v3[:, class_id].sum().item() - tp
    fn = confusion_v3[class_id, :].sum().item() - tp

    dice_den = 2 * tp + fp + fn
    iou_den = tp + fp + fn

    if dice_den == 0:
        print(f"{class_names[class_id]}: N/A")
        continue

    dice = 2 * tp / dice_den
    iou = tp / iou_den

    print(
        f"{class_names[class_id]}: "
        f"Dice={dice:.4f}, IoU={iou:.4f}"
    )

    if class_id != 0:
        dice_v3.append(dice)
        iou_v3.append(iou)

print("-" * 60)
print(f"Mean Tissue Dice: {np.mean(dice_v3):.4f}")
print(f"Mean Tissue IoU: {np.mean(iou_v3):.4f}")

# Analisis khusus Necrotic
necrotic_id = 4

gt_pixels = confusion_v3[necrotic_id, :].sum().item()
pred_pixels = confusion_v3[:, necrotic_id].sum().item()
correct_pixels = confusion_v3[necrotic_id, necrotic_id].item()

print("\nNECROTIC DETECTION")
print("Ground Truth:", gt_pixels)
print("Predicted:", pred_pixels)
print("Correct:", correct_pixels)


In [ ]:

v3_path = save_dir / "v3_epoch08.pth"

if not v3_path.exists():

    torch.save({
        "epoch": 8,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "train_loss": avg_loss_v3,
        "sampling": "necrotic_aware"
    }, v3_path)

print("V3 saved:", v3_path.exists())


In [ ]:

# Baca checkpoint V2
checkpoint_v2 = torch.load(
    save_dir / "v2_epoch07.pth",
    map_location=device,
    weights_only=True
)

# Restore bobot model
model.load_state_dict(
    checkpoint_v2["model_state"]
)

# Restore optimizer
optimizer.load_state_dict(
    checkpoint_v2["optimizer_state"]
)

# Atur random seed
torch.manual_seed(2026)
torch.cuda.manual_seed_all(2026)

print("Starting epoch:", checkpoint_v2["epoch"])
print("Control V2 ready!")


In [ ]:

# Aktifkan mode training
model.train()

control_total_loss = 0.0
control_total_samples = 0

# Gunakan DataLoader biasa, bukan V3
for images, masks in train_loader:

    images = normalize_images(images).to(device)
    masks = masks.to(device)

    optimizer.zero_grad(set_to_none=True)

    outputs = model(images)

    # Tetap menggunakan weighted loss V2
    loss = combined_loss_v2(outputs, masks)

    loss.backward()
    optimizer.step()

    batch_size = images.size(0)

    control_total_loss += loss.item() * batch_size
    control_total_samples += batch_size

avg_loss_control = control_total_loss / control_total_samples

print("CONTROL V2 - EPOCH 8")
print(f"Training Loss: {avg_loss_control:.4f}")
print(f"Samples processed: {control_total_samples}")

# Simpan checkpoint kontrol
torch.save({
    "epoch": 8,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "train_loss": avg_loss_control,
    "sampling": "standard"
}, save_dir / "v2_control_epoch08.pth")

print("Control checkpoint saved!")


In [ ]:

# Confusion matrix untuk V2 kontrol
confusion_control = torch.zeros(
    (NUM_CLASSES, NUM_CLASSES),
    dtype=torch.int64
)

model.eval()

with torch.no_grad():

    for images, masks in val_loader:

        # Normalisasi dan kirim ke GPU
        images = normalize_images(images).to(device)
        masks = masks.to(device)

        # Prediksi U-Net
        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        # Ambil piksel valid
        valid = (masks >= 0) & (masks < NUM_CLASSES)

        targets = masks[valid]
        preds = predictions[valid]

        # Hitung confusion matrix
        indices = targets * NUM_CLASSES + preds

        counts = torch.bincount(
            indices,
            minlength=NUM_CLASSES ** 2
        )

        confusion_control += counts.reshape(
            NUM_CLASSES, NUM_CLASSES
        ).cpu()

print("CONFUSION MATRIX - CONTROL V2 EPOCH 8")
print(confusion_control)


In [ ]:

dice_control = []
iou_control = []

print("VALIDATION METRICS - CONTROL V2 EPOCH 8")
print("-" * 60)

for class_id in range(NUM_CLASSES):

    tp = confusion_control[class_id, class_id].item()

    fp = confusion_control[:, class_id].sum().item() - tp
    fn = confusion_control[class_id, :].sum().item() - tp

    dice_den = 2 * tp + fp + fn
    iou_den = tp + fp + fn

    if dice_den == 0:
        print(f"{class_names[class_id]}: N/A")
        continue

    dice = 2 * tp / dice_den
    iou = tp / iou_den

    print(
        f"{class_names[class_id]}: "
        f"Dice={dice:.4f}, IoU={iou:.4f}"
    )

    if class_id != 0:
        dice_control.append(dice)
        iou_control.append(iou)

print("-" * 60)

print(f"Mean Tissue Dice: {np.mean(dice_control):.4f}")
print(f"Mean Tissue IoU: {np.mean(iou_control):.4f}")


In [ ]:

necrotic_id = 4

gt_pixels = confusion_control[necrotic_id, :].sum().item()
pred_pixels = confusion_control[:, necrotic_id].sum().item()
correct_pixels = confusion_control[necrotic_id, necrotic_id].item()

print("NECROTIC DETECTION - CONTROL V2")
print("-" * 40)
print("Ground Truth:", gt_pixels)
print("Predicted:", pred_pixels)
print("Correct:", correct_pixels)


In [ ]:

import numpy as np

necrotic_stats = []

for index in range(len(train_dataset)):

    # Ambil satu pasangan gambar dan mask
    image, mask = train_dataset[index]

    # Hitung piksel necrotic
    necrotic_pixels = (mask == 4).sum().item()

    if necrotic_pixels == 0:
        continue

    # Hitung seluruh piksel jaringan
    tissue_pixels = (
        (mask >= 1) & (mask <= 5)
    ).sum().item()

    # Persentase terhadap seluruh gambar
    image_percentage = (
        necrotic_pixels / mask.numel()
    ) * 100

    # Persentase terhadap area jaringan luka
    tissue_percentage = (
        necrotic_pixels / tissue_pixels
    ) * 100

    necrotic_stats.append({
        "index": index,
        "pixels": necrotic_pixels,
        "image_pct": image_percentage,
        "tissue_pct": tissue_percentage
    })

# Urutkan berdasarkan ukuran area necrotic
necrotic_stats.sort(
    key=lambda x: x["image_pct"]
)

# Tampilkan ringkasan
percentages = [
    item["image_pct"]
    for item in necrotic_stats
]

print("NECROTIC AREA ANALYSIS")
print("-" * 45)

print("Images with necrotic:", len(necrotic_stats))
print("Minimum area:", f"{min(percentages):.4f}%")
print("Median area:", f"{np.median(percentages):.4f}%")
print("Maximum area:", f"{max(percentages):.4f}%")

print("\nSmallest examples:")
for item in necrotic_stats[:3]:
    print(item)

print("\nLargest examples:")
for item in necrotic_stats[-3:]:
    print(item)


In [ ]:

# Load model V2 Control Epoch 8
checkpoint = torch.load(
    save_dir / "v2_control_epoch08.pth",
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state"])
model.eval()

# Pilih 3 contoh ukuran necrotic berbeda
sample_indices = [11, 49, 0]

fig, axes = plt.subplots(
    3, 3,
    figsize=(12, 12)
)

for row, index in enumerate(sample_indices):

    # Ambil gambar dan mask dari training
    image, mask = train_dataset[index]

    # Normalisasi gambar
    inputs = normalize_images(
        image.unsqueeze(0)
    ).to(device)

    # Prediksi model
    with torch.no_grad():
        outputs = model(inputs)
        prediction = outputs.argmax(dim=1)

    # Konversi ke NumPy
    image_np = image.permute(1, 2, 0).numpy()
    mask_np = mask.numpy()
    pred_np = prediction.squeeze(0).cpu().numpy()

    # Original
    axes[row, 0].imshow(image_np)
    axes[row, 0].set_title(f"Original - Index {index}")

    # Ground Truth Necrotic
    axes[row, 1].imshow(
        mask_np == 4,
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[row, 1].set_title("Ground Truth Necrotic")

    # Predicted Necrotic
    axes[row, 2].imshow(
        pred_np == 4,
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[row, 2].set_title("Predicted Necrotic")

    # Hitung jumlah piksel
    gt_count = np.sum(mask_np == 4)
    pred_count = np.sum(pred_np == 4)

    print(
        f"Index {index}: "
        f"GT={gt_count}, Predicted={pred_count}"
    )

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Analisis Probabilitas Necrotic
model.eval()

sample_indices = [11, 49, 0]

for index in sample_indices:
  image, mask = train_dataset[index]

  inputs = normalize_images(
      image.unsqueeze(0)
  ).to(device)

  # Hitung probabilitas
  with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)

  probabilities = probabilities[0].cpu()

  # Ambil kelas prediksi
  predictions = probabilities.argmax(dim=0)

  # Pilih area Ground Truth Necrotic
  necrotic_region = (mask == 4)

  # Prediksi khusus pada area tersebut
  predicted_classes = predictions[necrotic_region]

  counts = torch.bincount(
      predicted_classes,
      minlength=NUM_CLASSES
  )

  print(f"\nIMAGE INDEX {index}")
  print("-" * 45)
  print("Ground Truth Necrotic:",
        necrotic_region.sum().item())

  print("\nPredicted classes inside Necrotic:")
  for class_id in range(NUM_CLASSES):
    print(
            f"{class_names[class_id]}: "
            f"{counts[class_id].item()} pixels"
        )

    print("\nMean class probabilities:")

    for class_id in range(NUM_CLASSES):

        mean_prob = probabilities[
            class_id
        ][necrotic_region].mean().item()

        print(
            f"{class_names[class_id]}: "
            f"{mean_prob:.4f}"
        )

In [ ]:

# Ambil gambar dan mask dari training
image, mask = train_dataset[11]

# Cari koordinat piksel Necrotic
ys, xs = torch.where(mask == 4)

# Cari posisi tengah area Necrotic
center_y = int(torch.median(ys).item())
center_x = int(torch.median(xs).item())

# Tentukan ukuran patch
crop_size = 160

# Hitung batas crop agar tetap di dalam gambar
top = max(0, min(center_y - 80, 320 - crop_size))
left = max(0, min(center_x - 80, 320 - crop_size))

# Crop gambar dan mask pada posisi yang sama
image_crop = image[
    :,
    top:top + crop_size,
    left:left + crop_size
]

mask_crop = mask[
    top:top + crop_size,
    left:left + crop_size
]

print("Original image:", image.shape)
print("Cropped image:", image_crop.shape)

print("Original Necrotic pixels:", (mask == 4).sum().item())
print("Cropped Necrotic pixels:", (mask_crop == 4).sum().item())


In [ ]:

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(
    image.permute(1, 2, 0)
)
axes[0].set_title("Original")

axes[1].imshow(
    mask == 4,
    cmap="gray"
)
axes[1].set_title("Original Necrotic Mask")

axes[2].imshow(
    image_crop.permute(1, 2, 0)
)
axes[2].set_title("Focused Crop")

axes[3].imshow(
    mask_crop == 4,
    cmap="gray"
)
axes[3].set_title("Cropped Necrotic Mask")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

# Pilih gambar training index 11
entry = train_files[11]

# Pisahkan path foto dan mask
image_path, mask_path = entry.split()

# Buka file resolusi asli
original_image = Image.open(
    data_dir / image_path
).convert("RGB")

original_mask = Image.open(
    data_dir / mask_path
)

# Ubah mask menjadi NumPy
original_mask_array = np.array(original_mask)

# Cari koordinat jaringan Necrotic
ys, xs = np.where(original_mask_array == 4)

print("ORIGINAL IMAGE ANALYSIS")
print("-" * 40)

print("Image size:", original_image.size)
print("Mask size:", original_mask.size)

print("Necrotic pixels:", len(xs))

if len(xs) > 0:
    print("Necrotic bounding box:")
    print("X:", xs.min(), "-", xs.max())
    print("Y:", ys.min(), "-", ys.max())


In [ ]:

!git clone --depth 1 https://github.com/akabircs/WoundTissue.git data/WoundTissue


In [ ]:

from pathlib import Path

wound_dir = Path("data/WoundTissue")

print("Dataset ditemukan:", wound_dir.exists())

if wound_dir.exists():
    print("\nIsi folder:")
    for item in wound_dir.iterdir():
        print(item.name)


In [ ]:

from pathlib import Path
from collections import Counter

# Lokasi dataset baru
wound_dir = Path("data/WoundTissue")

# Periksa folder utama
print("Isi repository:")
for item in wound_dir.iterdir():
    print(item.name)

# Periksa jumlah file gambar dan label
for folder_name in ["image", "label"]:

    folder = wound_dir / folder_name

    files = [
        f for f in folder.rglob("*")
        if f.is_file()
    ]

    extensions = Counter(
        f.suffix.lower() for f in files
    )

    print(f"\nFolder: {folder_name}")
    print("Total files:", len(files))
    print("Extensions:", extensions)
    print("Sample files:", files[:3])


In [ ]:

from PIL import Image
import numpy as np
from collections import Counter

# Ambil file gambar dan mask
wt_images = sorted((wound_dir / "image").glob("*.png"))
wt_masks = sorted((wound_dir / "label").glob("*.png"))

# Periksa pasangan filename
image_names = {p.name for p in wt_images}
mask_names = {p.name for p in wt_masks}

print("Total images:", len(wt_images))
print("Total masks:", len(wt_masks))
print("Matching pairs:", len(image_names & mask_names))

# Analisis warna pada 3 mask pertama
for mask_path in wt_masks[:3]:

    mask = np.array(
        Image.open(mask_path).convert("RGB")
    )

    # Ubah setiap piksel menjadi tuple RGB
    pixels = mask.reshape(-1, 3)
    color_counts = Counter(map(tuple, pixels))

    print("\nMask:", mask_path.name)
    print("Shape:", mask.shape)
    print("Top RGB colors:")

    for color, count in color_counts.most_common(8):
        print(color, ":", count)


In [ ]:

necrotic_images = []
total_necrotic_pixels = 0

for mask_path in wt_masks:

    # Baca mask RGB
    mask = np.array(
        Image.open(mask_path).convert("RGB")
    )

    # Necrosis = hitam (0, 0, 0)
    necrotic_region = np.all(
        mask == [0, 0, 0],
        axis=2
    )

    # Hitung piksel necrotic
    necrotic_pixels = int(necrotic_region.sum())

    if necrotic_pixels > 0:

        necrotic_images.append(mask_path.name)

        total_necrotic_pixels += necrotic_pixels

        print(
            f"{mask_path.name}: "
            f"{necrotic_pixels:,} pixels"
        )

print("\nWOUNDTISSUE NECROTIC AUDIT")
print("-" * 40)

print("Total masks:", len(wt_masks))

print(
    "Images with Necrotic:",
    len(necrotic_images)
)

print(
    "Total Necrotic pixels:",
    total_necrotic_pixels
)


In [ ]:

!git clone --depth 1 https://github.com/uwm-bigdata/DFUTissueSegNet.git data/DFUTissueSegNet


In [ ]:

from pathlib import Path
from collections import Counter

dfu_dir = Path("data/DFUTissueSegNet")

print("Repository ditemukan:", dfu_dir.exists())

print("\nIsi folder utama:")
for item in dfu_dir.iterdir():
    print(item.name)

# Periksa folder dataset
dataset_dir = dfu_dir / "DFUTissue"

print("\nIsi folder DFUTissue:")
for item in dataset_dir.iterdir():
    print(item.name)

# Hitung format file
files = [
    f for f in dataset_dir.rglob("*")
    if f.is_file()
]

extensions = Counter(
    f.suffix.lower() for f in files
)

print("\nJumlah file:", len(files))
print("Format file:", extensions)

print("\nContoh file:")
for file in files[:10]:
    print(file)


In [ ]:

from pathlib import Path

labeled_dir = dataset_dir / "Labeled"
unlabeled_dir = dataset_dir / "Unlabeled"

# Periksa isi kedua folder
for folder in [labeled_dir, unlabeled_dir]:

    print(f"\nFOLDER: {folder.name}")
    print("-" * 45)

    # Hitung jumlah PNG
    png_files = sorted(folder.rglob("*.png"))

    print("Total PNG:", len(png_files))

    # Tampilkan struktur subfolder
    print("\nSubfolders:")
    for item in sorted(folder.iterdir()):
        if item.is_dir():
            print(" -", item.name)

    # Contoh lokasi file
    print("\nSample files:")
    for file in png_files[:5]:
        print(file.relative_to(dataset_dir))

# Periksa file TXT
print("\nTEXT FILES:")
for file in sorted(dataset_dir.rglob("*.txt")):
    print(file.relative_to(dataset_dir))


In [ ]:

labeled_dir = dataset_dir / "Labeled"

for version in ["Original", "Padded"]:

    folder = labeled_dir / version

    print(f"\n{version.upper()}")
    print("-" * 45)

    for path in sorted(folder.rglob("*")):

        if path.is_dir():

            count = len(list(path.glob("*.png")))

            if count > 0:
                print(
                    path.relative_to(labeled_dir),
                    ":", count, "PNG"
                )


In [ ]:

from PIL import Image
import numpy as np

# Ambil mask versi original
annotation_dir = labeled_dir / "Original" / "Annotations"

mask_paths = sorted(
    annotation_dir.rglob("*.png")
)

print("Total annotation files:", len(mask_paths))

# Periksa tiga mask pertama
for path in mask_paths[:3]:

    mask = np.array(
        Image.open(path).convert("RGB")
    )

    colors, counts = np.unique(
        mask.reshape(-1, 3),
        axis=0,
        return_counts=True
    )

    print("\nMask:", path.name)

    for color, count in zip(colors, counts):
        print(
            "RGB:", color.tolist(),
            "Pixels:", count
        )


In [ ]:

from collections import Counter

label_pixels = Counter()
label_images = Counter()

for path in mask_paths:

    mask = np.array(Image.open(path))

    # Jika mask RGB grayscale, ambil satu channel
    if mask.ndim == 3:
        assert np.array_equal(mask[:,:,0], mask[:,:,1])
        assert np.array_equal(mask[:,:,1], mask[:,:,2])
        mask = mask[:,:,0]

    labels, counts = np.unique(
        mask, return_counts=True
    )

    for label, count in zip(labels, counts):
        label_pixels[int(label)] += int(count)
        label_images[int(label)] += 1

print("DFUTISSUE LABEL AUDIT")
print("-" * 45)

for label in sorted(label_pixels):
    print(
        f"Label {label}: "
        f"{label_images[label]} images, "
        f"{label_pixels[label]:,} pixels"
    )


In [ ]:

# Pilih gambar training
index = 11

image_path, mask_path = train_files[index].split()

# Buka file resolusi asli
original_image = Image.open(
    data_dir / image_path
).convert("RGB")

original_mask = Image.open(
    data_dir / mask_path
)

mask_array = np.array(original_mask)

# Cari koordinat Necrotic
ys, xs = np.where(mask_array == 4)

# Tentukan titik tengah
center_x = int(np.median(xs))
center_y = int(np.median(ys))

# Ukuran patch dalam piksel asli
crop_size = min(
    320,
    original_image.width,
    original_image.height
)

# Tentukan posisi crop
left = max(
    0,
    min(center_x - crop_size // 2,
        original_image.width - crop_size)
)

top = max(
    0,
    min(center_y - crop_size // 2,
        original_image.height - crop_size)
)

# Crop image dan mask
crop_box = (
    left,
    top,
    left + crop_size,
    top + crop_size
)

image_patch = original_image.crop(crop_box)
mask_patch = original_mask.crop(crop_box)

print("Original size:", original_image.size)
print("Patch size:", image_patch.size)

print(
    "Original Necrotic:",
    np.sum(mask_array == 4)
)

print(
    "Patch Necrotic:",
    np.sum(np.array(mask_patch) == 4)
)


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].imshow(original_image)
axes[0].set_title("Original Image")

axes[1].imshow(image_patch)
axes[1].set_title("Native Resolution Patch")

axes[2].imshow(
    np.array(mask_patch) == 4,
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[2].set_title("Necrotic Ground Truth")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

# Gambar dan mask resolusi asli
photo = np.array(original_image)
mask = np.array(original_mask)

# Cari area berlabel Necrotic
necrotic_region = mask == 4

# Buat overlay transparan
overlay = np.zeros(
    (photo.shape[0], photo.shape[1], 4),
    dtype=np.uint8
)

# Hijau terang dengan transparansi
overlay[necrotic_region] = [0, 255, 0, 190]

# Cari batas area Necrotic
ys, xs = np.where(necrotic_region)

padding = 60

x_min = max(0, int(xs.min()) - padding)
x_max = min(photo.shape[1], int(xs.max()) + padding)

y_min = max(0, int(ys.min()) - padding)
y_max = min(photo.shape[0], int(ys.max()) + padding)

# Visualisasikan
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax in axes:
    ax.imshow(photo)
    ax.imshow(overlay)
    ax.axis("off")

axes[0].set_title("Necrotic Annotation Overlay")

axes[1].set_xlim(x_min, x_max)
axes[1].set_ylim(y_max, y_min)
axes[1].set_title("Zoom - Suspected Necrotic Area")

plt.tight_layout()
plt.show()


In [ ]:

# Identifikasi gambar yang sedang kita audit
entry = train_files[11]
image_path, mask_path = entry.split()

print("Patient:", Path(image_path).parent.name)
print("Image:", Path(image_path).name)
print("Mask:", Path(mask_path).name)

# Periksa folder gold standard
gold_dir = data_dir / "gold_standard"

print("\nIsi folder gold_standard:")

for item in sorted(gold_dir.iterdir()):
    print(item.name)


In [ ]:

import json

# Lokasi manifest
manifest_path = gold_dir / "manifest.jsonl"

# Baca seluruh record JSONL
with open(manifest_path, "r") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Total records:", len(records))

print("\nContoh record:")
for record in records[:2]:
    print(json.dumps(record, indent=2))


In [ ]:

# Lokasi anotasi klinisi
clinician_dir = gold_dir / "masks_by_clinician"

# Cari seluruh file anotasi
clinician_files = sorted(
    f for f in clinician_dir.rglob("*")
    if f.is_file()
)

print("Total clinician files:", len(clinician_files))

print("\nContoh lokasi file:")
for file in clinician_files[:10]:
    print(file.relative_to(gold_dir))


In [ ]:

# Lokasi anotasi klinisi
clinician_dir = gold_dir / "masks_by_clinician"

# Cari seluruh file anotasi
clinician_files = sorted(
    f for f in clinician_dir.rglob("*")
    if f.is_file()
)

print("Total clinician files:", len(clinician_files))

print("\nContoh lokasi file:")
for file in clinician_files[:10]:
    print(file.relative_to(gold_dir))


In [ ]:

# Gambar yang ingin kita audit
target_key = "Patient_11/P11T_4.jpg"

# Cari gambar dalam manifest
matches = [
    record for record in records
    if record["image_key"] == target_key
]

print("Matching records:", len(matches))

assert len(matches) == 1, "Record tidak ditemukan atau duplikat"

record = matches[0]

# Ambil path tissue mask dari 5 klinisi
tissue_keys = sorted([
    key for key in record
    if key.endswith("_tissue_mask")
])

print("\nJumlah klinisi:", len(tissue_keys))

for key in tissue_keys:
    print(key, ":", record[key])


In [ ]:

# Baca mask utama
main_mask = np.array(
    Image.open(
        data_dir / "Masks/Patient_11/P11T_4.png"
    )
)

# Area yang ditandai Necrotic pada mask utama
necrotic_region = main_mask == 4

print("Main mask Necrotic pixels:",
      necrotic_region.sum())

print("\nCLINICIAN AGREEMENT")
print("-" * 55)

for key in tissue_keys:

    # Baca mask dari masing-masing klinisi
    path = data_dir / record[key]

    clinician_mask = np.array(Image.open(path))

    assert clinician_mask.shape == main_mask.shape

    # Ambil label pada area Necrotic mask utama
    labels = clinician_mask[necrotic_region]

    unique, counts = np.unique(
        labels,
        return_counts=True
    )

    print(f"\n{key.split('_tissue_mask')[0]}")

    for label, count in zip(unique, counts):
        name = class_names.get(
            int(label),
            f"Unknown ({label})"
        )

        print(f"{name}: {count} pixels")


In [ ]:

# Gabungkan mask dari lima klinisi
clinician_masks = []

for key in tissue_keys:

    path = data_dir / record[key]

    clinician_mask = np.array(
        Image.open(path)
    )

    clinician_masks.append(clinician_mask)

# Bentuk tensor: [5, Height, Width]
stacked_masks = np.stack(
    clinician_masks,
    axis=0
)

# Area yang diberi label Necrotic pada mask utama
region = main_mask == 4

# Hitung suara setiap kelas pada area ini
slough_votes = (
    stacked_masks[:, region] == 2
).sum(axis=0)

necrotic_votes = (
    stacked_masks[:, region] == 4
).sum(axis=0)

# Hitung hasil mayoritas
majority_slough = np.sum(slough_votes >= 3)
majority_necrotic = np.sum(necrotic_votes >= 3)

print("ANNOTATION AGREEMENT AUDIT")
print("-" * 45)

print("Total pixels:", region.sum())
print("Majority Slough:", majority_slough)
print("Majority Necrotic:", majority_necrotic)

print("\nVote distribution:")
for votes in range(6):
    count = np.sum(slough_votes == votes)

    if count > 0:
        print(f"{votes}/5 Slough votes: {count} pixels")


In [ ]:

# Buat indeks berdasarkan nama file gambar
gold_lookup = {
    record["image_file_name"]: record
    for record in records
}

# Simpan gambar training yang mengandung Necrotic
necrotic_train_images = []

for entry in train_files:

    image_path, mask_path = entry.split()

    # Baca mask resolusi asli
    mask = np.array(
        Image.open(data_dir / mask_path)
    )

    # Periksa keberadaan Necrotic
    if np.any(mask == 4):
        necrotic_train_images.append(image_path)

# Cari yang tersedia dalam gold standard
available = [
    path for path in necrotic_train_images
    if path in gold_lookup
]

missing = [
    path for path in necrotic_train_images
    if path not in gold_lookup
]

print("NECROTIC MULTI-CLINICIAN AUDIT")
print("-" * 50)

print("Total Necrotic training images:",
      len(necrotic_train_images))

print("With multi-clinician annotations:",
      len(available))

print("Without multi-clinician annotations:",
      len(missing))

print("\nAVAILABLE FOR AUDIT:")
for path in available:
    print(path)


In [ ]:

# Buat pencocokan foto dengan mask utama
train_mask_lookup = {}

for entry in train_files:
    image_path, mask_path = entry.split()
    train_mask_lookup[image_path] = mask_path

# Fungsi membaca mask
def read_mask(path):
    mask = np.array(Image.open(path))

    assert mask.ndim == 2, "Mask harus berisi ID kelas"

    return mask

# Daftar label yang mungkin digunakan
label_ids = [0, 1, 2, 3, 4, 5, 255]

print("Data siap untuk audit!")
print("Jumlah gambar:", len(available))


In [ ]:

audit_results = []

for image_path in available:

    # Ambil record gold standard
    record = gold_lookup[image_path]

    # Baca mask utama
    main_path = train_mask_lookup[image_path]
    main = read_mask(data_dir / main_path)

    # Pilih area Necrotic dari mask utama
    region = main == 4

    # Baca lima anotasi klinisi
    tissue_keys = sorted(
        key for key in record
        if key.endswith("_tissue_mask")
    )

    masks = [
        read_mask(data_dir / record[key])
        for key in tissue_keys
    ]

    assert all(m.shape == main.shape for m in masks)

    stacked = np.stack(masks, axis=0)

    # Hitung suara setiap kelas di setiap piksel
    votes = np.stack([
        (stacked == label).sum(axis=0)
        for label in label_ids
    ])

    # Ambil kelas dengan suara terbanyak
    max_votes = votes.max(axis=0)

    majority_label = np.array(label_ids)[
        votes.argmax(axis=0)
    ]

    # Mayoritas minimal 3 dari 5 klinisi
    has_majority = max_votes >= 3

    # Hitung hasil khusus area Necrotic utama
    necrotic_majority = np.sum(
        region & has_majority & (majority_label == 4)
    )

    slough_majority = np.sum(
        region & has_majority & (majority_label == 2)
    )

    other_majority = np.sum(
        region & has_majority &
        (majority_label != 2) &
        (majority_label != 4)
    )

    no_majority = np.sum(
        region & ~has_majority
    )

    audit_results.append({
        "image": image_path,
        "total": int(region.sum()),
        "necrotic": int(necrotic_majority),
        "slough": int(slough_majority),
        "other": int(other_majority),
        "no_majority": int(no_majority)
    })

print("Audit selesai:", len(audit_results), "gambar")


In [ ]:

print("NECROTIC ANNOTATION AGREEMENT")
print("-" * 65)

for result in audit_results:

    total = result["total"]

    print("\nImage:", result["image"])
    print("Main Necrotic Pixels:", total)

    for key in ["necrotic", "slough", "other", "no_majority"]:

        count = result[key]
        percentage = count / total * 100

        print(
            f"{key}: {count} pixels "
            f"({percentage:.2f}%)"
        )

print("\nAUDIT SUMMARY")
print("-" * 45)

total_pixels = sum(r["total"] for r in audit_results)
necrotic_pixels = sum(r["necrotic"] for r in audit_results)
slough_pixels = sum(r["slough"] for r in audit_results)

print("Total main Necrotic pixels:", total_pixels)
print("Majority Necrotic:", necrotic_pixels)
print("Majority Slough:", slough_pixels)


In [ ]:

# Semua label LUTSeg
label_names = {
    0: "Background",
    1: "Epithelial",
    2: "Slough",
    3: "Granulation",
    4: "Necrotic",
    5: "Other",
    255: "Ignore"
}

label_ids = list(label_names.keys())

for image_path in available:

    record = gold_lookup[image_path]

    # Baca mask utama
    main = read_mask(
        data_dir / train_mask_lookup[image_path]
    )

    # Area Necrotic pada mask utama
    region = main == 4

    # Baca lima mask klinisi
    keys = sorted(
        k for k in record
        if k.endswith("_tissue_mask")
    )

    masks = np.stack([
        read_mask(data_dir / record[k])
        for k in keys
    ])

    # Ambil suara hanya pada area Necrotic
    region_votes = masks[:, region]

    # Hitung voting setiap kelas
    votes = np.stack([
        (region_votes == label).sum(axis=0)
        for label in label_ids
    ])

    max_votes = votes.max(axis=0)

    majority_labels = np.array(label_ids)[
        votes.argmax(axis=0)
    ]

    print("\nIMAGE:", image_path)
    print("Necrotic pixels:", region.sum())
    print("-" * 45)

    # Tampilkan mayoritas tiap kelas
    for label, name in label_names.items():

        count = np.sum(
            (majority_labels == label) &
            (max_votes >= 3)
        )

        if count > 0:
            print(f"{name}: {count:,} pixels")

    # Piksel tanpa mayoritas
    uncertain = np.sum(max_votes < 3)

    print("No majority:", uncertain)


In [ ]:

for image_path in available:

    record = gold_lookup[image_path]

    # Baca mask utama LUTSeg
    main = read_mask(
        data_dir / train_mask_lookup[image_path]
    )

    print("\nIMAGE:", image_path)
    print("-" * 50)

    results = []

    for key in sorted(record):

        if not key.endswith("_tissue_mask"):
            continue

        # Baca anotasi klinisi
        clinician = read_mask(
            data_dir / record[key]
        )

        # Pastikan ukuran cocok
        assert main.shape == clinician.shape

        # Hitung persentase piksel yang identik
        matching = np.mean(
            main == clinician
        ) * 100

        # Apakah seluruh mask benar-benar identik?
        exact_match = np.array_equal(
            main, clinician
        )

        results.append((
            key.split("_tissue_mask")[0],
            matching,
            exact_match
        ))

    # Urutkan berdasarkan kesamaan
    results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    for clinician, similarity, exact in results:

        print(
            f"{clinician}: "
            f"{similarity:.2f}% | "
            f"Exact match: {exact}"
        )


In [ ]:

# Pilih gambar yang sudah kita audit
image_path = "Images/Patient_11/P11T_4.jpg"

record = gold_lookup[image_path]

# Baca mask referensi
main = read_mask(
    data_dir / train_mask_lookup[image_path]
)

# Ambil lima anotasi klinisi
tissue_keys = sorted(
    key for key in record
    if key.endswith("_tissue_mask")
)

clinician_masks = np.stack([
    read_mask(data_dir / record[key])
    for key in tissue_keys
])

# Hitung dukungan untuk label referensi per piksel
agreement_count = (
    clinician_masks == main[None, :, :]
).sum(axis=0)

# Fokus pada area Necrotic
necrotic_region = main == 4

print("REFERENCE LABEL AGREEMENT")
print("-" * 45)

for votes in range(6):

    count = np.sum(
        agreement_count[necrotic_region] == votes
    )

    if count > 0:
        print(
            f"{votes}/5 clinicians: "
            f"{count} pixels"
        )

# Hitung piksel dengan dukungan < 3 klinisi
low_agreement = np.sum(
    necrotic_region & (agreement_count < 3)
)

print("\nLow agreement pixels:", low_agreement)
print("Total Necrotic:", necrotic_region.sum())


In [ ]:

print("NECROTIC AGREEMENT SUMMARY")
print("-" * 65)

for image_path in available:

    record = gold_lookup[image_path]

    # Mask utama
    main = read_mask(
        data_dir / train_mask_lookup[image_path]
    )

    # Lima mask klinisi
    keys = sorted(
        k for k in record
        if k.endswith("_tissue_mask")
    )

    masks = np.stack([
        read_mask(data_dir / record[k])
        for k in keys
    ])

    # Hanya piksel Necrotic pada mask utama
    region = main == 4

    # Jumlah klinisi yang memilih Necrotic
    support = (masks[:, region] == 4).sum(axis=0)

    # Kelompok kesepakatan
    low = np.sum(support <= 2)
    moderate = np.sum(support == 3)
    high = np.sum(support >= 4)

    total = region.sum()

    print("\n", image_path.split("/")[-1])
    print("Total:", total)

    print(f"Low: {low / total * 100:.2f}%")
    print(f"Moderate: {moderate / total * 100:.2f}%")
    print(f"High: {high / total * 100:.2f}%")

    assert low + moderate + high == total


In [ ]:

# Pilih gambar dengan disagreement terbesar
image_path = "Images/Patient_1/P1_T1.jpeg"

record = gold_lookup[image_path]

# Foto asli
image = np.array(
    Image.open(data_dir / image_path).convert("RGB")
)

# Mask utama
main = read_mask(
    data_dir / train_mask_lookup[image_path]
)

# Lima anotasi klinisi
keys = sorted(
    k for k in record
    if k.endswith("_tissue_mask")
)

masks = np.stack([
    read_mask(data_dir / record[k])
    for k in keys
])

# Hitung dukungan klinisi untuk label utama
support = (masks == main[None, :, :]).sum(axis=0)

# Tampilkan hanya area Necrotic
necrotic_region = main == 4

agreement_map = np.ma.masked_where(
    ~necrotic_region,
    support
)

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(image)
axes[0].set_title("Original Wound Image")
axes[0].axis("off")

axes[1].imshow(image)

heatmap = axes[1].imshow(
    agreement_map,
    cmap="RdYlGn",
    vmin=1,
    vmax=5,
    alpha=0.8
)

axes[1].set_title("Necrotic Agreement Map")
axes[1].axis("off")

colorbar = fig.colorbar(
    heatmap,
    ax=axes[1],
    ticks=[1, 2, 3, 4, 5],
    fraction=0.046
)

colorbar.set_label("Clinicians agreeing with reference")

plt.tight_layout()
plt.show()


In [ ]:

# Siapkan weight map default
weight_map = np.ones(
    main.shape,
    dtype=np.float32
)

# Area jaringan, bukan background atau ignore
tissue_region = (main >= 1) & (main <= 5)

# Bobot berdasarkan dukungan klinisi
weight_map[tissue_region & (support <= 2)] = 0.50

weight_map[tissue_region & (support == 3)] = 0.75

weight_map[tissue_region & (support >= 4)] = 1.00

# Ignore pixel tidak digunakan saat training
weight_map[main == 255] = 0.0

# Periksa bobot khusus area Necrotic
necrotic_weights = weight_map[main == 4]

values, counts = np.unique(
    necrotic_weights,
    return_counts=True
)

print("NECROTIC LOSS WEIGHT DISTRIBUTION")
print("-" * 45)

for value, count in zip(values, counts):
    print(f"Weight {value:.2f}: {count:,} pixels")

print("\nWeight map shape:", weight_map.shape)
print("Average Necrotic weight:", necrotic_weights.mean())

# Pastikan background tidak berubah
assert np.all(weight_map[main == 0] == 1.0)


In [ ]:

def make_agreement_weight_map(image_path):

    # Pastikan hanya gambar training
    if image_path not in train_mask_lookup:
        raise ValueError("Image is not in training split")

    # Baca mask utama
    main = read_mask(
        data_dir / train_mask_lookup[image_path]
    )

    # Semua piksel memiliki bobot netral
    weights = np.ones(
        main.shape,
        dtype=np.float32
    )

    # Ignore tidak berkontribusi pada loss
    weights[main == 255] = 0.0

    # Jika tidak ada anotasi multi-klinisi,
    # gunakan bobot netral
    if image_path not in gold_lookup:
        return weights

    record = gold_lookup[image_path]

    keys = sorted(
        key for key in record
        if key.endswith("_tissue_mask")
    )

    assert len(keys) == 5

    # Hitung jumlah klinisi yang setuju
    # dengan label utama
    support = np.zeros(
        main.shape,
        dtype=np.uint8
    )

    for key in keys:

        clinician = read_mask(
            data_dir / record[key]
        )

        assert clinician.shape == main.shape

        support += (clinician == main)

    # Hanya bobot piksel jaringan
    tissue = (main >= 1) & (main <= 5)

    weights[tissue & (support <= 2)] = 0.50
    weights[tissue & (support == 3)] = 0.75
    weights[tissue & (support >= 4)] = 1.00

    return weights


In [ ]:

# Uji pada P1_T1
test_image = "Images/Patient_1/P1_T1.jpeg"

new_weights = make_agreement_weight_map(
    test_image
)

print("Shape:", new_weights.shape)
print("Unique weights:", np.unique(new_weights))

# Bandingkan dengan Praktik 68
print(
    "Matches previous result:",
    np.allclose(new_weights, weight_map)
)

# Uji pada P11T_4
test_image_2 = "Images/Patient_11/P11T_4.jpg"

weights_2 = make_agreement_weight_map(
    test_image_2
)

main_2 = read_mask(
    data_dir / train_mask_lookup[test_image_2]
)

print("\nP11T_4 Necrotic weights:")
print(
    np.unique(
        weights_2[main_2 == 4],
        return_counts=True
    )
)


In [ ]:

class WeightedWoundDataset(Dataset):

    def __init__(self, base_dataset, file_list):
        self.base_dataset = base_dataset
        self.file_list = file_list

        assert len(base_dataset) == len(file_list)

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):

        # Ambil image dan mask dari dataset lama
        image, mask = self.base_dataset[idx]

        # Cari path gambar berdasarkan index
        image_path, _ = self.file_list[idx].split()

        # Buat agreement weight map resolusi asli
        weight = make_agreement_weight_map(image_path)

        # Resize mengikuti ukuran mask hasil dataset
        height, width = mask.shape

        weight_image = Image.fromarray(weight)

        weight_image = weight_image.resize(
            (width, height),
            resample=Image.Resampling.NEAREST
        )

        # Konversi menjadi tensor float32
        weight_tensor = torch.from_numpy(
            np.array(weight_image, dtype=np.float32)
        )

        return image, mask, weight_tensor


In [ ]:

# Buat dataset baru
weighted_train_dataset = WeightedWoundDataset(
    train_dataset,
    train_files
)

# Uji pada gambar index 11 (P11T_4)
image, mask, weight = weighted_train_dataset[11]

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)
print("Weight shape:", weight.shape)

print("\nImage dtype:", image.dtype)
print("Mask dtype:", mask.dtype)
print("Weight dtype:", weight.dtype)

print("\nUnique weights:", torch.unique(weight))

# Pastikan dimensi mask dan bobot sama
assert mask.shape == weight.shape

# Pastikan piksel ignore mendapat bobot nol
assert torch.all(weight[mask == 255] == 0)

# Cek bobot pada Necrotic
necrotic_weights = weight[mask == 4]

print("\nNecrotic pixels:", len(necrotic_weights))
print("Necrotic weights:", torch.unique(necrotic_weights))


In [ ]:

import torch.nn.functional as F

def pixel_weighted_ce(outputs, targets, pixel_weights):

    # Ambil class weights dari loss V2
    class_w = weighted_ce_loss.weight.to(outputs.device)

    # Tandai piksel yang boleh digunakan
    valid = targets != 255

    # Hitung Cross-Entropy per piksel
    ce_map = F.cross_entropy(
        outputs,
        targets,
        weight=class_w,
        ignore_index=255,
        reduction="none"
    )

    # Hindari indexing class 255
    safe_targets = targets.masked_fill(~valid, 0)

    # Gabungkan bobot kelas dan bobot piksel
    denominator = (
        class_w[safe_targets]
        * pixel_weights
        * valid
    ).sum()

    # Hitung loss akhir
    numerator = (ce_map * pixel_weights).sum()

    loss = numerator / denominator.clamp_min(1e-8)

    return loss


In [ ]:

# Ambil dua contoh dari dataset
samples = [
    weighted_train_dataset[i]
    for i in [0, 11]
]

# Susun menjadi batch
images = torch.stack(
    [s[0] for s in samples]
).to(device)

masks = torch.stack(
    [s[1] for s in samples]
).to(device)

weights = torch.stack(
    [s[2] for s in samples]
).to(device)

# Prediksi tanpa training
model.eval()

with torch.no_grad():

    outputs = model(
        normalize_images(images)
    )

    # Loss V2
    old_loss = weighted_ce_loss(
        outputs, masks
    )

    # Loss baru dengan semua pixel weight = 1
    unit_loss = pixel_weighted_ce(
        outputs,
        masks,
        torch.ones_like(weights)
    )

    # Loss baru dengan agreement weights
    aware_loss = pixel_weighted_ce(
        outputs,
        masks,
        weights
    )

print("LOSS COMPARISON")
print("-" * 45)

print("V2 Weighted CE:", old_loss.item())
print("V4 Unit Weights:", unit_loss.item())
print("V4 Agreement Aware:", aware_loss.item())

print("\nV2 == V4 Unit Weights:",
      torch.allclose(
          old_loss,
          unit_loss,
          rtol=1e-5,
          atol=1e-5
      ))


In [ ]:

# Combined loss untuk eksperimen V4
def combined_loss_v4(outputs, targets, pixel_weights):

    ce = pixel_weighted_ce(
        outputs,
        targets,
        pixel_weights
    )

    dice = dice_loss(
        outputs,
        targets
    )

    return ce + dice


# Bandingkan dengan loss V2
with torch.no_grad():

    loss_v2 = combined_loss_v2(
        outputs,
        masks
    )

    loss_v4_unit = combined_loss_v4(
        outputs,
        masks,
        torch.ones_like(weights)
    )

    loss_v4_aware = combined_loss_v4(
        outputs,
        masks,
        weights
    )


print("COMBINED LOSS VERIFICATION")
print("-" * 45)

print("V2 Combined Loss:", loss_v2.item())
print("V4 Unit Weights:", loss_v4_unit.item())
print("V4 Agreement Aware:", loss_v4_aware.item())

print(
    "\nV2 == V4 Unit Weights:",
    torch.allclose(
        loss_v2,
        loss_v4_unit,
        rtol=1e-5,
        atol=1e-5
    )
)


In [ ]:

# DataLoader untuk training V4
train_loader_v4 = DataLoader(
    weighted_train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# Ambil satu batch
batch_images, batch_masks, batch_weights = next(
    iter(train_loader_v4)
)

print("V4 DATALOADER TEST")
print("-" * 45)

print("Images:", batch_images.shape)
print("Masks:", batch_masks.shape)
print("Weights:", batch_weights.shape)

print("\nWeight values:", torch.unique(batch_weights))

# Validasi bentuk data
assert batch_images.shape[1] == 3
assert batch_masks.shape == batch_weights.shape
assert batch_masks.dtype == torch.int64
assert batch_weights.dtype == torch.float32

print("\nDataLoader V4: PASS")


In [ ]:

print("ANNOTATION-AWARE DATASET COVERAGE")
print("-" * 50)

gold_count = 0
modified_count = 0

for idx, entry in enumerate(train_files):

    image_path, _ = entry.split()

    # Cek apakah ada anotasi multi-klinisi
    if image_path in gold_lookup:
        gold_count += 1

    # Ambil mask dan bobot hasil resize
    _, mask, weight = weighted_train_dataset[idx]

    # Piksel valid dengan bobot tidak netral
    modified_pixels = (
        (weight != 1.0) &
        (mask != 255)
    ).sum().item()

    if modified_pixels > 0:

        modified_count += 1

        print(
            f"{image_path}: "
            f"{modified_pixels:,} modified pixels"
        )

print("\nSUMMARY")
print("-" * 50)

print("Total training images:", len(train_files))
print("With gold annotations:", gold_count)
print("With modified weights:", modified_count)


In [ ]:

# Lokasi checkpoint V2 Epoch 7
checkpoint_path = Path(
    "/content/drive/MyDrive/WoundVisionAI/v2_epoch07.pth"
)

print("CHECKPOINT INSPECTION")
print("-" * 50)

print("File exists:", checkpoint_path.exists())

assert checkpoint_path.exists(), "Checkpoint tidak ditemukan"

# Baca checkpoint tanpa mengubah model aktif
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):

    print("\nTop-level keys:")

    for key in list(checkpoint.keys())[:15]:
        print("-", key)

    if "epoch" in checkpoint:
        print("\nSaved epoch:", checkpoint["epoch"])


In [ ]:

import random
from torch.utils.data import Subset

# Seed eksperimen
SEED = 2026

# Buat urutan gambar acak yang tetap
generator = torch.Generator().manual_seed(SEED)

fixed_indices = torch.randperm(
    len(weighted_train_dataset),
    generator=generator
).tolist()

# Dataset dengan urutan tetap
fixed_dataset = Subset(
    weighted_train_dataset,
    fixed_indices
)

# DataLoader untuk kedua eksperimen
fixed_train_loader = DataLoader(
    fixed_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("CONTROLLED TRAINING SETUP")
print("-" * 45)
print("Training images:", len(fixed_dataset))
print("Batches:", len(fixed_train_loader))
print("Seed:", SEED)
print("First 5 indices:", fixed_indices[:5])


In [ ]:

def reset_experiment():

    # Reset random seed
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # Kembalikan model ke Epoch 7
    model.load_state_dict(
        checkpoint["model_state"]
    )

    # Kembalikan optimizer ke Epoch 7
    optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )


def train_experiment(name, use_agreement):

    # Kondisi awal harus sama
    reset_experiment()

    model.train()
    total_loss = 0.0
    total_samples = 0

    for images, masks, weights in fixed_train_loader:

        images = images.to(device)
        masks = masks.to(device)
        weights = weights.to(device)

        # Normalisasi gambar
        images = normalize_images(images)

        # Forward pass
        outputs = model(images)

        # Pilih fungsi loss
        if use_agreement:
            loss = combined_loss_v4(
                outputs, masks, weights
            )
        else:
            loss = combined_loss_v2(
                outputs, masks
            )

        # Backpropagation
        optimizer.zero_grad(set_to_none=True)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    average_loss = total_loss / total_samples

    # Simpan checkpoint eksperimen
    save_path = Path(
        f"/content/drive/MyDrive/WoundVisionAI/{name}.pth"
    )

    torch.save({
        "epoch": 8,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "train_loss": average_loss,
        "seed": SEED,
        "use_agreement": use_agreement
    }, save_path)

    print(f"\n{name}")
    print("Training Loss:", round(average_loss, 4))
    print("Saved:", save_path)

    return average_loss


In [ ]:

# Eksperimen A: Baseline
loss_a = train_experiment(
    name="v4_control_epoch08",
    use_agreement=False
)

# Eksperimen B: Annotation-Aware
loss_b = train_experiment(
    name="v4_aware_epoch08",
    use_agreement=True
)

print("\nEXPERIMENT COMPLETED")
print("-" * 45)
print("V4-A Control Loss:", round(loss_a, 4))
print("V4-B Aware Loss:", round(loss_b, 4))


In [ ]:

def evaluate_checkpoint(checkpoint_path):

    # Baca checkpoint
    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    # Muat model yang ingin dievaluasi
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    # Baris = ground truth
    # Kolom = prediksi model
    confusion = torch.zeros(
        (6, 6),
        dtype=torch.int64
    )

    with torch.inference_mode():

        for images, masks in val_loader:

            images = images.to(device)
            masks = masks.to(device)

            # Forward pass
            outputs = model(
                normalize_images(images)
            )

            predictions = outputs.argmax(dim=1)

            # Abaikan piksel ignore
            valid = masks != 255

            # Kode pasangan label dan prediksi
            pairs = (
                masks[valid] * 6
                + predictions[valid]
            )

            # Akumulasi confusion matrix
            batch_confusion = torch.bincount(
                pairs,
                minlength=36
            ).reshape(6, 6)

            confusion += batch_confusion.cpu()

    # True Positive per kelas
    tp = confusion.diag().float()

    # Jumlah piksel aktual setiap kelas
    actual = confusion.sum(dim=1).float()

    # Jumlah piksel prediksi setiap kelas
    predicted = confusion.sum(dim=0).float()

    # Hitung metrik
    dice = 2 * tp / (actual + predicted).clamp_min(1)

    iou = tp / (
        actual + predicted - tp
    ).clamp_min(1)

    precision = tp / predicted.clamp_min(1)
    recall = tp / actual.clamp_min(1)

    # Macro Dice untuk kelas jaringan 1–5
    macro_dice = dice[1:6].mean().item()

    return {
        "confusion": confusion,
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "actual": actual,
        "predicted": predicted,
        "tp": tp,
        "macro_dice": macro_dice
    }

print("Evaluation function ready!")


In [ ]:

# Lokasi checkpoint
save_dir = Path(
    "/content/drive/MyDrive/WoundVisionAI"
)

# Evaluasi kedua model
results_a = evaluate_checkpoint(
    save_dir / "v4_control_epoch08.pth"
)

results_b = evaluate_checkpoint(
    save_dir / "v4_aware_epoch08.pth"
)

# Nama kelas
names = [
    "Background",
    "Epithelial",
    "Slough",
    "Granulation",
    "Necrotic",
    "Other"
]

print("V4 VALIDATION COMPARISON")
print("-" * 65)

print(
    f"{'Class':<15}"
    f"{'Dice A':>12}"
    f"{'Dice B':>12}"
    f"{'Difference':>14}"
)

for i, name in enumerate(names):

    dice_a = results_a["dice"][i].item()
    dice_b = results_b["dice"][i].item()

    print(
        f"{name:<15}"
        f"{dice_a:>12.4f}"
        f"{dice_b:>12.4f}"
        f"{dice_b - dice_a:>+14.4f}"
    )

print("\nMACRO TISSUE DICE")
print("-" * 45)

print("V4-A:", round(results_a["macro_dice"], 4))
print("V4-B:", round(results_b["macro_dice"], 4))

print("\nNECROTIC DETAIL")
print("-" * 45)

for label, result in [
    ("V4-A Control", results_a),
    ("V4-B Aware", results_b)
]:

    print(f"\n{label}")

    print(
        "Actual Necrotic:",
        int(result["actual"][4].item())
    )

    print(
        "Predicted Necrotic:",
        int(result["predicted"][4].item())
    )

    print(
        "Correct Necrotic:",
        int(result["tp"][4].item())
    )

    print(
        "Precision:",
        round(result["precision"][4].item(), 4)
    )

    print(
        "Recall:",
        round(result["recall"][4].item(), 4)
    )

    print(
        "Dice:",
        round(result["dice"][4].item(), 4)
    )

    print(
        "IoU:",
        round(result["iou"][4].item(), 4)
    )


In [ ]:

# Nama kelas jaringan
class_names_list = [
    "Background",
    "Epithelial",
    "Slough",
    "Granulation",
    "Necrotic",
    "Other"
]

# Bandingkan kesalahan dua model
for experiment, result in [
    ("V4-A Control", results_a),
    ("V4-B Annotation-Aware", results_b)
]:

    cm = result["confusion"]

    print("\n" + "=" * 55)
    print(experiment)
    print("=" * 55)

    # Periksa ground truth Slough dan Necrotic
    for class_id in [2, 4]:

        print(
            "\nACTUAL:",
            class_names_list[class_id]
        )

        total = cm[class_id].sum().item()

        # Lihat diprediksi sebagai kelas apa
        for pred_id, count in enumerate(cm[class_id]):

            count = count.item()

            percentage = count / total * 100

            print(
                f"Predicted {class_names_list[pred_id]:<12}"
                f": {count:>7,} pixels "
                f"({percentage:>6.2f}%)"
            )


In [ ]:

checkpoints_to_compare = {
    "V2 Epoch 7": save_dir / "v2_epoch07.pth",
    "V2 Old Control Epoch 8": save_dir / "v2_control_epoch08.pth",
    "V4-A Control": save_dir / "v4_control_epoch08.pth",
    "V4-B Aware": save_dir / "v4_aware_epoch08.pth"
}

print("CHECKPOINT COMPARISON")
print("-" * 70)

for name, path in checkpoints_to_compare.items():

    if not path.exists():
        print(name, "- File not found")
        continue

    result = evaluate_checkpoint(path)

    print(f"\n{name}")
    print("Slough Dice:", round(result["dice"][2].item(), 4))
    print("Necrotic Dice:", round(result["dice"][4].item(), 4))

    print(
        "Predicted Slough:",
        int(result["predicted"][2].item())
    )

    print(
        "Predicted Necrotic:",
        int(result["predicted"][4].item())
    )

    print(
        "Macro Tissue Dice:",
        round(result["macro_dice"], 4)
    )


In [ ]:

print("VALIDATION DATA AUDIT")
print("-" * 50)

print("Validation files:", len(val_files))
print("Validation dataset:", len(val_loader.dataset))
print("Validation batches:", len(val_loader))

print("\nFirst validation file:")
print(val_files[0])

print("\nLast validation file:")
print(val_files[-1])

# Hitung piksel ground truth tanpa model
gt_counts = torch.zeros(6, dtype=torch.int64)
ignore_count = 0

for images, masks in val_loader:

    valid = masks != 255

    gt_counts += torch.bincount(
        masks[valid].reshape(-1),
        minlength=6
    )

    ignore_count += (masks == 255).sum().item()

print("\nGROUND TRUTH PIXEL COUNTS")

for i, count in enumerate(gt_counts):
    print(f"{names[i]}: {count.item():,}")

print("\nIgnore pixels:", ignore_count)

# Bandingkan dengan evaluasi V4
print(
    "\nMatches V4 confusion matrix:",
    torch.equal(
        gt_counts,
        results_a["actual"].long()
    )
)


In [ ]:

# Checkpoint yang hasil lamanya berbeda
path = save_dir / "v2_control_epoch08.pth"

ckpt = torch.load(
    path,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(ckpt["model_state"])
model.eval()

# Fokus Slough dan Necrotic
stats = {
    2: {"tp": 0, "actual": 0, "predicted": 0},
    4: {"tp": 0, "actual": 0, "predicted": 0}
}

with torch.inference_mode():

    for images, masks in val_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(
            normalize_images(images)
        )

        predictions = outputs.argmax(dim=1)

        valid = masks != 255

        for class_id in [2, 4]:

            actual = (masks == class_id) & valid
            predicted = (predictions == class_id) & valid

            stats[class_id]["tp"] += (
                actual & predicted
            ).sum().item()

            stats[class_id]["actual"] += actual.sum().item()

            stats[class_id]["predicted"] += predicted.sum().item()

print("INDEPENDENT DICE VERIFICATION")
print("-" * 50)

for class_id in [2, 4]:

    s = stats[class_id]

    denominator = s["actual"] + s["predicted"]

    dice = (
        2 * s["tp"] / denominator
        if denominator > 0 else 0.0
    )

    print(f"\nClass: {names[class_id]}")
    print("Actual:", s["actual"])
    print("Predicted:", s["predicted"])
    print("Correct:", s["tp"])
    print("Dice:", round(dice, 4))


In [ ]:

from datetime import datetime

checkpoints = {
    "V2 Epoch 7": "v2_epoch07.pth",
    "V2 Old Control Epoch 8": "v2_control_epoch08.pth",
    "V4-A Control": "v4_control_epoch08.pth",
    "V4-B Aware": "v4_aware_epoch08.pth"
}

print("CHECKPOINT PROVENANCE AUDIT")
print("-" * 60)

for name, filename in checkpoints.items():

    path = save_dir / filename

    if not path.exists():
        print(f"\n{name}: NOT FOUND")
        continue

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    modified = datetime.fromtimestamp(
        path.stat().st_mtime
    )

    print(f"\n{name}")
    print("Filename:", filename)
    print("Saved epoch:", ckpt.get("epoch"))
    print("Train loss:", ckpt.get("train_loss"))
    print("Seed:", ckpt.get("seed", "Not recorded"))
    print("Agreement:", ckpt.get("use_agreement", "Not recorded"))
    print("File modified:", modified)
    print("File size MB:", round(
        path.stat().st_size / (1024 ** 2), 2
    ))


In [ ]:

print("CHECKPOINT CONSISTENCY AUDIT")
print("-" * 55)

# Checkpoint yang tersimpan dalam memori
memory_checkpoint = checkpoint

# Baca ulang checkpoint dari Google Drive
disk_checkpoint = torch.load(
    save_dir / "v2_epoch07.pth",
    map_location="cpu",
    weights_only=True
)

print("V2 EPOCH 7")
print(
    "Memory loss:",
    memory_checkpoint.get("train_loss")
)

print(
    "Disk loss:",
    disk_checkpoint.get("train_loss")
)

# Bandingkan seluruh parameter model
same_weights = all(
    torch.equal(
        memory_checkpoint["model_state"][key].cpu(),
        disk_checkpoint["model_state"][key].cpu()
    )
    for key in memory_checkpoint["model_state"]
)

print("Model weights identical:", same_weights)

# Bandingkan nilai loss V4 dalam memori
# dengan checkpoint yang sekarang tersimpan
print("\nV4 EXPERIMENTS")
print("-" * 55)

for name, filename, memory_loss in [
    ("V4-A", "v4_control_epoch08.pth", loss_a),
    ("V4-B", "v4_aware_epoch08.pth", loss_b)
]:

    saved = torch.load(
        save_dir / filename,
        map_location="cpu",
        weights_only=True
    )

    disk_loss = saved["train_loss"]

    print(f"\n{name}")
    print("Memory loss:", memory_loss)
    print("Disk loss:", disk_loss)

    print(
        "Loss matches:",
        abs(memory_loss - disk_loss) < 1e-6
    )


In [ ]:

import hashlib
import json
import uuid
from datetime import datetime, timezone

def file_sha256(path):

    hasher = hashlib.sha256()

    with open(path, "rb") as f:
        while chunk := f.read(1024 * 1024):
            hasher.update(chunk)

    return hasher.hexdigest()


# Identitas audit yang unik
audit_id = "audit_" + uuid.uuid4().hex[:8]

# File checkpoint yang ingin dicatat
checkpoint_files = [
    "v2_epoch07.pth",
    "v2_control_epoch08.pth",
    "v4_control_epoch08.pth",
    "v4_aware_epoch08.pth"
]

manifest = {
    "audit_id": audit_id,
    "created_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "dataset": "LUTSeg",
    "train_images": len(train_files),
    "validation_images": len(val_files),
    "image_size": [320, 320],
    "checkpoints": {}
}

for filename in checkpoint_files:

    path = save_dir / filename

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    manifest["checkpoints"][filename] = {
        "epoch": ckpt["epoch"],
        "train_loss": ckpt.get("train_loss"),
        "sha256": file_sha256(path)
    }

# Simpan manifest tanpa menimpa file lama
manifest_path = save_dir / f"{audit_id}.json"

with open(manifest_path, "x") as f:
    json.dump(manifest, f, indent=2)

print("EXPERIMENT MANIFEST CREATED")
print("-" * 50)

print("Audit ID:", audit_id)
print("Saved:", manifest_path)

for filename, info in manifest["checkpoints"].items():

    print("\n", filename)
    print("Epoch:", info["epoch"])
    print("Loss:", info["train_loss"])
    print("SHA256:", info["sha256"][:16], "...")


In [ ]:

def save_checkpoint_safe(filename, payload):

    # Tentukan lokasi penyimpanan
    path = save_dir / filename

    # Pastikan folder tersedia
    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # Mode "xb" = exclusive binary write
    # Gagal jika file sudah ada
    with path.open("xb") as f:
        torch.save(payload, f)

    print("Checkpoint saved:", path)

    return path


In [ ]:

# Buat nama file test yang unik
test_name = (
    "checkpoint_test_"
    + uuid.uuid4().hex[:8]
    + ".pth"
)

# Payload kecil untuk pengujian
test_payload = {
    "epoch": 1,
    "loss": 0.5,
    "purpose": "overwrite_protection_test"
}

# Percobaan pertama: simpan file
test_path = save_checkpoint_safe(
    test_name,
    test_payload
)

# Catat hash sebelum percobaan kedua
hash_before = file_sha256(test_path)

# Percobaan kedua: gunakan nama file yang sama
try:
    save_checkpoint_safe(
        test_name,
        test_payload
    )

    print("ERROR: Overwrite protection failed")

except FileExistsError:
    print("PASS: Existing checkpoint protected")

# Pastikan isi file tidak berubah
hash_after = file_sha256(test_path)

print(
    "SHA256 unchanged:",
    hash_before == hash_after
)

assert hash_before == hash_after


In [ ]:

# Audit resolusi asli gambar training
image_sizes = []
size_mismatches = []

for entry in train_files:

    image_path, mask_path = entry.split()

    with Image.open(data_dir / image_path) as image:
        image_size = image.size

    with Image.open(data_dir / mask_path) as mask:
        mask_size = mask.size

    # Periksa kesesuaian dimensi
    if image_size != mask_size:
        size_mismatches.append(image_path)

    width, height = image_size

    image_sizes.append({
        "path": image_path,
        "width": width,
        "height": height
    })

# Cari gambar yang lebih kecil dari patch 320
small_images = [
    item for item in image_sizes
    if item["width"] < 320 or item["height"] < 320
]

print("NATIVE RESOLUTION AUDIT")
print("-" * 50)

print("Total images:", len(image_sizes))

print("Minimum width:",
      min(x["width"] for x in image_sizes))

print("Minimum height:",
      min(x["height"] for x in image_sizes))

print("Maximum width:",
      max(x["width"] for x in image_sizes))

print("Maximum height:",
      max(x["height"] for x in image_sizes))

print("\nImage-mask size mismatches:",
      len(size_mismatches))

print("Images smaller than 320:",
      len(small_images))

print("\nFive smallest images:")

for item in sorted(
    image_sizes,
    key=lambda x: x["width"] * x["height"]
)[:5]:

    print(
        item["path"],
        item["width"],
        "x",
        item["height"]
    )


In [ ]:

def crop_wound_patch(
    image_path,
    mask_path,
    patch_size=320,
    focus_class=None,
    rng=None
):
    if rng is None:
        rng = np.random.default_rng()

    # Baca gambar dan mask resolusi asli
    image = np.array(
        Image.open(data_dir / image_path).convert("RGB")
    )

    mask = np.array(
        Image.open(data_dir / mask_path)
    )

    assert image.shape[:2] == mask.shape

    height, width = mask.shape

    # Hitung kebutuhan padding
    pad_h = max(0, patch_size - height)
    pad_w = max(0, patch_size - width)

    top_pad = pad_h // 2
    bottom_pad = pad_h - top_pad

    left_pad = pad_w // 2
    right_pad = pad_w - left_pad

    # Padding gambar: ulangi piksel tepi
    image = np.pad(
        image,
        (
            (top_pad, bottom_pad),
            (left_pad, right_pad),
            (0, 0)
        ),
        mode="edge"
    )

    # Padding mask: gunakan ignore label 255
    mask = np.pad(
        mask,
        (
            (top_pad, bottom_pad),
            (left_pad, right_pad)
        ),
        mode="constant",
        constant_values=255
    )

    height, width = mask.shape

    # Pilih lokasi crop
    if focus_class is not None and np.any(mask == focus_class):

        ys, xs = np.where(mask == focus_class)

        idx = int(rng.integers(len(ys)))

        # Lokasi target berada di dalam crop
        offset_y = patch_size // 2
        offset_x = patch_size // 2

        top = int(np.clip(
            ys[idx] - offset_y,
            0,
            height - patch_size
        ))

        left = int(np.clip(
            xs[idx] - offset_x,
            0,
            width - patch_size
        ))

    else:

        # Random crop biasa
        top = int(rng.integers(
            0, height - patch_size + 1
        ))

        left = int(rng.integers(
            0, width - patch_size + 1
        ))

    # Crop image dan mask dengan koordinat sama
    image_patch = image[
        top:top + patch_size,
        left:left + patch_size
    ]

    mask_patch = mask[
        top:top + patch_size,
        left:left + patch_size
    ]

    return image_patch, mask_patch


In [ ]:

rng = np.random.default_rng(2026)

# TEST A: Gambar dengan lebar 288
small_entry = next(
    entry for entry in train_files
    if "Patient_34/P34T_1.jpg" in entry
)

image_path, mask_path = small_entry.split()

image_a, mask_a = crop_wound_patch(
    image_path,
    mask_path,
    rng=rng
)

print("TEST A — SMALL IMAGE")
print("Image shape:", image_a.shape)
print("Mask shape:", mask_a.shape)
print("Ignore pixels:", np.sum(mask_a == 255))

# TEST B: Gambar Necrotic Patient 11
image_path, mask_path = train_files[11].split()

image_b, mask_b = crop_wound_patch(
    image_path,
    mask_path,
    focus_class=4,
    rng=rng
)

print("\nTEST B — NECROTIC PATCH")
print("Image shape:", image_b.shape)
print("Mask shape:", mask_b.shape)
print("Necrotic pixels:", np.sum(mask_b == 4))

# Validasi
assert image_a.shape == (320, 320, 3)
assert mask_a.shape == (320, 320)

assert image_b.shape == (320, 320, 3)
assert mask_b.shape == (320, 320)
assert np.any(mask_b == 4)

print("\nPATCH TEST: PASS")


In [ ]:

# Gunakan gambar Patient 11 yang sama
image_path, mask_path = train_files[11].split()

# Metode A: Full-image resize
full_image = Image.open(
    data_dir / image_path
).convert("RGB").resize(
    (320, 320),
    Image.Resampling.BILINEAR
)

full_mask = Image.open(
    data_dir / mask_path
).resize(
    (320, 320),
    Image.Resampling.NEAREST
)

full_image = np.array(full_image)
full_mask = np.array(full_mask)

# Metode B: Patch dari Praktik 95
patch_image = image_b
patch_mask = mask_b

# Fungsi untuk menampilkan anotasi Necrotic
def show_necrotic_overlay(ax, image, mask, title):

    ax.imshow(image)

    overlay = np.zeros(
        (mask.shape[0], mask.shape[1], 4),
        dtype=np.float32
    )

    # Warna hijau untuk Necrotic
    overlay[mask == 4] = [0, 1, 0, 0.75]

    ax.imshow(overlay)

    ax.set_title(title)
    ax.axis("off")


# Bandingkan hasil
fig, axes = plt.subplots(
    1, 2, figsize=(12, 6)
)

show_necrotic_overlay(
    axes[0],
    full_image,
    full_mask,
    "A — Full Image Resize"
)

show_necrotic_overlay(
    axes[1],
    patch_image,
    patch_mask,
    "B — Native Resolution Patch"
)

plt.tight_layout()
plt.show()

print("NECROTIC PIXEL COMPARISON")
print("-" * 45)

print(
    "Full resize:",
    np.sum(full_mask == 4)
)

print(
    "Native patch:",
    np.sum(patch_mask == 4)
)


In [ ]:

# Gunakan gambar Patient 11 yang sama
image_path, mask_path = train_files[11].split()

# Ambil patch dengan posisi target di tengah
center_image, center_mask = crop_wound_patch(
    image_path,
    mask_path,
    focus_class=4,
    rng=np.random.default_rng(2026)
)

# Bandingkan secara visual
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

show_necrotic_overlay(
    axes[0],
    image_b,
    mask_b,
    "Before - Random Target Position"
)

show_necrotic_overlay(
    axes[1],
    center_image,
    center_mask,
    "After - Centered Target"
)

plt.tight_layout()
plt.show()

# Hitung proporsi jaringan dalam patch
def tissue_percentage(mask):
    tissue = np.isin(mask, [1, 2, 3, 4, 5])
    return tissue.mean() * 100

print("PATCH QUALITY COMPARISON")
print("-" * 45)

print(
    "Old patch tissue:",
    round(tissue_percentage(mask_b), 2), "%"
)

print(
    "Centered patch tissue:",
    round(tissue_percentage(center_mask), 2), "%"
)

print(
    "Centered Necrotic pixels:",
    np.sum(center_mask == 4)
)


In [ ]:

class WoundDatasetV5(Dataset):

    def __init__(
        self,
        file_list,
        patch_size=320,
        focus_prob=0.5
    ):
        self.file_list = file_list
        self.patch_size = patch_size
        self.focus_prob = focus_prob

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):

        image_path, mask_path = (
            self.file_list[idx].split()
        )

        # Random generator untuk crop
        # Seed berasal dari PyTorch
        seed = torch.randint(
            0, 2**31 - 1, (1,)
        ).item()

        rng = np.random.default_rng(seed)

        focus_class = None

        # 50% peluang tissue-focused crop
        if rng.random() < self.focus_prob:

            original_mask = np.array(
                Image.open(data_dir / mask_path)
            )

            # Fokus sementara kelas 1-3
            candidates = [
                c for c in [1, 2, 3]
                if np.any(original_mask == c)
            ]

            if candidates:
                focus_class = int(
                    rng.choice(candidates)
                )

        # Ambil patch resolusi asli
        image_patch, mask_patch = crop_wound_patch(
            image_path,
            mask_path,
            patch_size=self.patch_size,
            focus_class=focus_class,
            rng=rng
        )

        # Konversi image menjadi tensor CHW
        image_tensor = torch.from_numpy(
            image_patch.copy()
        ).permute(2, 0, 1).float() / 255.0

        # Mask tetap integer kelas
        mask_tensor = torch.from_numpy(
            mask_patch.copy()
        ).long()

        return image_tensor, mask_tensor


In [ ]:

# Buat dataset baru
train_dataset_v5 = WoundDatasetV5(
    train_files,
    patch_size=320,
    focus_prob=0.5
)

print("V5 DATASET TEST")
print("-" * 45)

print("Dataset length:", len(train_dataset_v5))

# Uji gambar Patient 11
image, mask = train_dataset_v5[11]

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)

print("Image dtype:", image.dtype)
print("Mask dtype:", mask.dtype)

# Uji variasi patch pada gambar yang sama
print("\nPATCH VARIATION TEST")

tissue_ratios = []
necrotic_counts = []

for i in range(10):

    image, mask = train_dataset_v5[11]

    mask_np = mask.numpy()

    tissue_ratios.append(
        tissue_percentage(mask_np)
    )

    necrotic_counts.append(
        int(np.sum(mask_np == 4))
    )

print("Tissue percentages:")
print([round(x, 2) for x in tissue_ratios])

print("\nNecrotic pixel counts:")
print(necrotic_counts)

# Validasi bentuk tensor
assert image.shape == (3, 320, 320)
assert mask.shape == (320, 320)

assert image.dtype == torch.float32
assert mask.dtype == torch.int64

print("\nV5 DATASET: PASS")


In [ ]:

# Nama kelas
class_labels = [
    "Background",
    "Epithelial",
    "Slough",
    "Granulation",
    "Necrotic",
    "Other"
]

# Penampung jumlah piksel
baseline_counts = np.zeros(6, dtype=np.int64)
patch_counts = np.zeros(6, dtype=np.int64)

ignore_pixels = 0
empty_patches = 0
necrotic_patches = 0

# Gunakan seed tetap untuk audit
# agar sampling dapat diulang
with torch.random.fork_rng(devices=[]):

    torch.manual_seed(2026)

    for idx in range(len(train_dataset_v5)):

        # Dataset baseline
        _, baseline_mask = train_dataset[idx]

        # Dataset V5
        _, patch_mask = train_dataset_v5[idx]

        # Konversi ke NumPy
        baseline_np = baseline_mask.numpy()
        patch_np = patch_mask.numpy()

        # Hitung piksel setiap kelas
        for class_id in range(6):

            baseline_counts[class_id] += np.sum(
                baseline_np == class_id
            )

            patch_counts[class_id] += np.sum(
                patch_np == class_id
            )

        # Hitung ignore pixels
        ignore_pixels += np.sum(patch_np == 255)

        # Periksa patch tanpa jaringan
        tissue = (patch_np >= 1) & (patch_np <= 5)

        if not np.any(tissue):
            empty_patches += 1

        # Periksa patch dengan Necrotic
        if np.any(patch_np == 4):
            necrotic_patches += 1

print("V5 CLASS DISTRIBUTION AUDIT")
print("-" * 65)

baseline_total = baseline_counts.sum()
patch_total = patch_counts.sum()

print(
    f"{'Class':<15}"
    f"{'Baseline %':>15}"
    f"{'V5 Patch %':>15}"
)

for i, name in enumerate(class_labels):

    baseline_pct = (
        baseline_counts[i] / baseline_total * 100
    )

    patch_pct = (
        patch_counts[i] / patch_total * 100
    )

    print(
        f"{name:<15}"
        f"{baseline_pct:>14.2f}%"
        f"{patch_pct:>14.2f}%"
    )

print("\nPATCH STATISTICS")
print("-" * 45)

print("Total patches:", len(train_dataset_v5))
print("Ignore pixels:", ignore_pixels)
print("Patches without tissue:", empty_patches)
print("Patches with Necrotic:", necrotic_patches)

print(
    "Valid tissue percentage:",
    round(
        patch_counts[1:6].sum()
        / patch_total * 100, 2
    ),
    "%"
)


In [ ]:

# Seed yang sama dengan Praktik 101
PATCH_SEED = 2026

# Simpan hasil cropping ke memori
with torch.random.fork_rng(devices=[]):

    torch.manual_seed(PATCH_SEED)

    frozen_v5_samples = [
        train_dataset_v5[i]
        for i in range(len(train_dataset_v5))
    ]

print("FROZEN V5 DATASET")
print("-" * 50)

print("Total frozen patches:", len(frozen_v5_samples))

# Hitung ulang distribusi kelas
frozen_counts = np.zeros(6, dtype=np.int64)

for image, mask in frozen_v5_samples:

    mask_np = mask.numpy()

    valid = mask_np != 255

    frozen_counts += np.bincount(
        mask_np[valid].reshape(-1),
        minlength=6
    )[:6]

# Bandingkan dengan audit Praktik 101
same_distribution = np.array_equal(
    frozen_counts,
    patch_counts
)

print("Same distribution as audit:",
      same_distribution)

# Pastikan image dan mask tetap sesuai
image, mask = frozen_v5_samples[0]

print("\nFirst image:", image.shape)
print("First mask:", mask.shape)

assert len(frozen_v5_samples) == 111
assert same_distribution
assert image.shape == (3, 320, 320)
assert mask.shape == (320, 320)

print("\nFROZEN DATASET: PASS")


In [ ]:

# Buat nama snapshot unik
snapshot_name = (
    f"v5_frozen_seed{PATCH_SEED}_"
    f"{uuid.uuid4().hex[:8]}.pt"
)

# Informasi yang akan disimpan
snapshot = {
    "seed": PATCH_SEED,
    "train_files": list(train_files),
    "samples": frozen_v5_samples,
    "class_counts": frozen_counts.tolist(),
    "patch_size": 320
}

# Simpan tanpa menimpa file lain
snapshot_path = save_checkpoint_safe(
    snapshot_name,
    snapshot
)

print("\nFROZEN DATASET SAVED")
print("-" * 50)

print("File:", snapshot_path.name)
print("Total samples:", len(frozen_v5_samples))
print("SHA256:", file_sha256(snapshot_path)[:16])


In [ ]:

from torch.utils.data import Subset

# Urutan tetap untuk kedua eksperimen
TRAIN_SEED = 2026

generator = torch.Generator().manual_seed(
    TRAIN_SEED
)

order = torch.randperm(
    len(train_files),
    generator=generator
).tolist()

# Control: full-image resize
v5_control_dataset = Subset(
    train_dataset,
    order
)

# Experimental: frozen native patches
v5_patch_dataset = Subset(
    frozen_v5_samples,
    order
)

# DataLoader memakai urutan tetap
v5_control_loader = DataLoader(
    v5_control_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

v5_patch_loader = DataLoader(
    v5_patch_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("V5 CONTROLLED DATALOADERS")
print("-" * 50)

print("Control samples:", len(v5_control_dataset))
print("Patch samples:", len(v5_patch_dataset))

print("Control batches:", len(v5_control_loader))
print("Patch batches:", len(v5_patch_loader))

print("First 5 indices:", order[:5])
print("Matches old order:", order == fixed_indices)

# Pastikan jumlah batch identik
assert len(v5_control_loader) == 28
assert len(v5_patch_loader) == 28

print("\nV5 DATALOADERS: PASS")


In [ ]:

# Cari patch yang memiliki padding
test_idx = next(
    i for i, (_, mask) in enumerate(frozen_v5_samples)
    if torch.any(mask == 255)
)

image, mask = frozen_v5_samples[test_idx]

images = image.unsqueeze(0).to(device)
masks = mask.unsqueeze(0).to(device)

model.eval()

with torch.no_grad():

    outputs = model(
        normalize_images(images)
    )

    # Loss menggunakan prediksi asli
    original_loss = combined_loss_v2(
        outputs,
        masks
    )

    # Ubah logits hanya pada piksel padding
    ignore_region = (masks == 255).unsqueeze(1)

    modified_outputs = torch.where(
        ignore_region,
        torch.full_like(outputs, 20.0),
        outputs
    )

    # Hitung ulang loss
    modified_loss = combined_loss_v2(
        modified_outputs,
        masks
    )

print("V5 IGNORE INDEX TEST")
print("-" * 45)

print("Test sample:", test_idx)
print("Ignore pixels:", (masks == 255).sum().item())

print("Original loss:", original_loss.item())
print("Modified loss:", modified_loss.item())

same_loss = torch.allclose(
    original_loss,
    modified_loss,
    atol=1e-6,
    rtol=1e-6
)

print("\nIgnore pixels excluded:", same_loss)

assert torch.isfinite(original_loss)
assert same_loss

print("V5 LOSS TEST: PASS")


In [ ]:

# Checkpoint awal yang sama
start_checkpoint = torch.load(
    save_dir / "v2_epoch07.pth",
    map_location="cpu",
    weights_only=True
)

def train_v5(name, loader):

    # Reset random seed
    random.seed(2026)
    np.random.seed(2026)
    torch.manual_seed(2026)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(2026)

    # Reset model dan optimizer
    model.load_state_dict(
        start_checkpoint["model_state"]
    )

    optimizer.load_state_dict(
        start_checkpoint["optimizer_state"]
    )

    model.train()

    total_loss = 0.0
    total_samples = 0

    for images, masks in loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(
            normalize_images(images)
        )

        loss = combined_loss_v2(
            outputs,
            masks
        )

        optimizer.zero_grad(set_to_none=True)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    average_loss = total_loss / total_samples

    # Nama unik untuk mencegah overwrite
    filename = (
        f"{name}_seed2026_"
        f"{uuid.uuid4().hex[:8]}.pth"
    )

    path = save_checkpoint_safe(
        filename,
        {
            "epoch": 8,
            "experiment": name,
            "seed": 2026,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "train_loss": average_loss,
            "start_checkpoint": "v2_epoch07.pth",
            "sample_order": order
        }
    )

    print(f"\n{name}")
    print("Training loss:", round(average_loss, 4))
    print("Checkpoint:", path.name)

    return path, average_loss


# A: Full image resize
v5_a_path, v5_a_loss = train_v5(
    "v5_full_resize",
    v5_control_loader
)

# B: Native resolution patch
v5_b_path, v5_b_loss = train_v5(
    "v5_native_patch",
    v5_patch_loader
)

print("\nV5 TRAINING COMPLETED")
print("-" * 45)

print("V5-A loss:", round(v5_a_loss, 4))
print("V5-B loss:", round(v5_b_loss, 4))


In [ ]:

# Evaluasi checkpoint V5-A
results_v5a = evaluate_checkpoint(
    v5_a_path
)

# Evaluasi checkpoint V5-B
results_v5b = evaluate_checkpoint(
    v5_b_path
)

print("V5 VALIDATION COMPARISON")
print("-" * 65)

print(
    f"{'Class':<15}"
    f"{'V5-A Dice':>12}"
    f"{'V5-B Dice':>12}"
    f"{'Difference':>14}"
)

for i, name in enumerate(names):

    dice_a = results_v5a["dice"][i].item()
    dice_b = results_v5b["dice"][i].item()

    print(
        f"{name:<15}"
        f"{dice_a:>12.4f}"
        f"{dice_b:>12.4f}"
        f"{dice_b - dice_a:>+14.4f}"
    )

print("\nMACRO TISSUE DICE")
print("-" * 45)

print(
    "V5-A:",
    round(results_v5a["macro_dice"], 4)
)

print(
    "V5-B:",
    round(results_v5b["macro_dice"], 4)
)


In [ ]:

print("V5 NECROTIC PERFORMANCE")
print("-" * 55)

for name, result in [
    ("V5-A Full Resize", results_v5a),
    ("V5-B Native Patch", results_v5b)
]:

    print(f"\n{name}")

    print(
        "Actual Necrotic:",
        int(result["actual"][4].item())
    )

    print(
        "Predicted Necrotic:",
        int(result["predicted"][4].item())
    )

    print(
        "Correct Necrotic:",
        int(result["tp"][4].item())
    )

    print(
        "Precision:",
        round(result["precision"][4].item(), 4)
    )

    print(
        "Recall:",
        round(result["recall"][4].item(), 4)
    )

    print(
        "Dice:",
        round(result["dice"][4].item(), 4)
    )

print("\nSLOUGH PERFORMANCE")
print("-" * 55)

for name, result in [
    ("V5-A", results_v5a),
    ("V5-B", results_v5b)
]:

    print(
        name,
        "| Dice:",
        round(result["dice"][2].item(), 4),
        "| Recall:",
        round(result["recall"][2].item(), 4)
    )


In [ ]:

# Lokasi kedua checkpoint control
path_v4 = save_dir / "v4_control_epoch08.pth"
path_v5 = v5_a_path

# Baca checkpoint
ckpt_v4 = torch.load(
    path_v4,
    map_location="cpu",
    weights_only=True
)

ckpt_v5 = torch.load(
    path_v5,
    map_location="cpu",
    weights_only=True
)

# Ambil parameter dan buffer model
state_v4 = ckpt_v4["model_state"]
state_v5 = ckpt_v5["model_state"]

# Pastikan struktur sama
assert state_v4.keys() == state_v5.keys()

# Cari tensor yang berbeda
different_keys = []

for key in state_v4:

    if not torch.equal(
        state_v4[key],
        state_v5[key]
    ):
        different_keys.append(key)

print("MODEL CHECKPOINT COMPARISON")
print("-" * 50)

print("V4 train loss:", ckpt_v4["train_loss"])
print("V5 train loss:", ckpt_v5["train_loss"])

print("Total tensors:", len(state_v4))
print("Different tensors:", len(different_keys))

print(
    "Models identical:",
    len(different_keys) == 0
)

print("\nFirst different tensors:")

for key in different_keys[:5]:
    print("-", key)


In [ ]:

print("CONTROL CHECKPOINT RE-EVALUATION")
print("-" * 55)

for name, path in [
    ("V4-A Control", path_v4),
    ("V5-A Full Resize", path_v5)
]:

    result = evaluate_checkpoint(path)

    print(f"\n{name}")

    print(
        "Slough Dice:",
        round(result["dice"][2].item(), 4)
    )

    print(
        "Necrotic Dice:",
        round(result["dice"][4].item(), 4)
    )

    print(
        "Necrotic Recall:",
        round(result["recall"][4].item(), 4)
    )

    print(
        "Macro Tissue Dice:",
        round(result["macro_dice"], 4)
    )


In [ ]:
# Membuat posisi sliding windows
def get_tile_starts(length, patch_size=320, stride=160):
  if length <= patch_size:
    return[0]

  positions = list(
      range(0, length - patch_size + 1, stride)
  )

  # Pastikan ujung gambar ikut tercakup
  last_position = length - patch_size

  if positions[-1] != last_position:
    positions.append(last_position)

  return positions

# Pilih gambar validation dengan resolusi terkecil
val_sizes = []

for entry in val_files:

    image_path, mask_path = entry.split()

    with Image.open(data_dir / image_path) as img:
        width, height = img.size

    val_sizes.append(
        (width * height, image_path, mask_path)
    )

_, test_image_path, test_mask_path = min(val_sizes)

# Baca foto asli, tanpa resize
test_image = np.array(
    Image.open(data_dir / test_image_path).convert("RGB")
)

height, width = test_image.shape[:2]

# Padding jika lebih kecil dari 320
pad_h = max(0, 320 - height)
pad_w = max(0, 320 - width)

padded_image = np.pad(
    test_image,
    ((0, pad_h), (0, pad_w), (0, 0)),
    mode="edge"
)

H, W = padded_image.shape[:2]

ys = get_tile_starts(H)
xs = get_tile_starts(W)

tile_positions = [
    (y, x)
    for y in ys
    for x in xs
]

print("SLIDING WINDOW SETUP")
print("-" * 45)

print("Image:", test_image_path)
print("Original shape:", test_image.shape)
print("Padded shape:", padded_image.shape)
print("Total patches:", len(tile_positions))
print("Grid:", len(ys), "rows x", len(xs), "columns")


In [ ]:

# Buat coverage map
coverage = np.zeros(
    (H, W),
    dtype=np.int32
)

# Hitung berapa patch mencakup setiap piksel
for y, x in tile_positions:

    coverage[
        y:y + 320,
        x:x + 320
    ] += 1

print("SLIDING WINDOW COVERAGE")
print("-" * 45)

print("Minimum coverage:", coverage.min())
print("Maximum coverage:", coverage.max())

print(
    "Uncovered pixels:",
    np.sum(coverage == 0)
)

# Visualisasi
plt.figure(figsize=(8, 5))

plt.imshow(coverage, cmap="viridis")
plt.colorbar(label="Number of overlapping patches")

plt.title("Sliding Window Coverage Map")
plt.axis("off")

plt.show()

assert np.all(coverage > 0)

print("SLIDING WINDOW: PASS")


In [ ]:

def sliding_window_predict(
    image,
    model,
    patch_size=320,
    stride=160
):
    # Ukuran foto asli
    original_h, original_w = image.shape[:2]

    # Padding jika foto lebih kecil dari patch
    pad_h = max(0, patch_size - original_h)
    pad_w = max(0, patch_size - original_w)

    padded = np.pad(
        image,
        ((0, pad_h), (0, pad_w), (0, 0)),
        mode="edge"
    )

    H, W = padded.shape[:2]

    # Koordinat semua patch
    ys = get_tile_starts(H, patch_size, stride)
    xs = get_tile_starts(W, patch_size, stride)

    # Penampung probabilitas setiap kelas
    prob_sum = np.zeros(
        (6, H, W),
        dtype=np.float32
    )

    # Hitung jumlah prediksi per piksel
    coverage = np.zeros(
        (H, W),
        dtype=np.float32
    )

    model.eval()

    with torch.inference_mode():

        for y in ys:
            for x in xs:

                # Ambil patch RGB
                patch = padded[
                    y:y + patch_size,
                    x:x + patch_size
                ]

                # Konversi menjadi tensor BCHW
                tensor = torch.from_numpy(
                    patch.copy()
                ).permute(2, 0, 1)

                tensor = (
                    tensor.float()
                    .unsqueeze(0)
                    .to(device) / 255.0
                )

                # Normalisasi ImageNet
                tensor = normalize_images(tensor)

                # Forward pass
                logits = model(tensor)

                # Probabilitas enam kelas
                probs = torch.softmax(
                    logits,
                    dim=1
                )[0].cpu().numpy()

                # Tambahkan probabilitas patch
                prob_sum[
                    :,
                    y:y + patch_size,
                    x:x + patch_size
                ] += probs

                coverage[
                    y:y + patch_size,
                    x:x + patch_size
                ] += 1

    # Pastikan seluruh piksel tercakup
    assert np.all(coverage > 0)

    # Rata-rata probabilitas pada overlap
    avg_probs = prob_sum / coverage[None, :, :]

    # Hapus area padding
    avg_probs = avg_probs[
        :,
        :original_h,
        :original_w
    ]

    # Pilih kelas dengan probabilitas tertinggi
    prediction = np.argmax(
        avg_probs,
        axis=0
    ).astype(np.uint8)

    return prediction, avg_probs


In [ ]:

# Muat checkpoint V5-B
ckpt = torch.load(
    v5_b_path,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(
    ckpt["model_state"]
)

model.eval()

# Gunakan foto validation dari Praktik 111
prediction, probabilities = sliding_window_predict(
    test_image,
    model,
    patch_size=320,
    stride=160
)

print("SLIDING WINDOW INFERENCE")
print("-" * 45)

print("Original image:", test_image.shape)
print("Prediction:", prediction.shape)
print("Probabilities:", probabilities.shape)

print("\nPredicted classes:")
for class_id, name in enumerate(names):

    count = np.sum(prediction == class_id)

    print(f"{name}: {count:,} pixels")

# Pemeriksaan hasil
assert prediction.shape == test_image.shape[:2]

assert probabilities.shape == (
    6,
    test_image.shape[0],
    test_image.shape[1]
)

assert np.allclose(
    probabilities.sum(axis=0),
    1.0,
    atol=1e-5
)

print("\nSLIDING WINDOW INFERENCE: PASS")


In [ ]:

# Baca ground truth resolusi asli
gt_mask = np.array(
    Image.open(data_dir / test_mask_path)
)

assert gt_mask.shape == prediction.shape

print("GROUND TRUTH VS PREDICTION")
print("-" * 55)

for class_id, name in enumerate(names):

    actual = np.sum(gt_mask == class_id)
    predicted = np.sum(prediction == class_id)

    print(
        f"{name:<14}"
        f"GT: {actual:>8,}"
        f" | Pred: {predicted:>8,}"
    )

# Periksa rata-rata probabilitas tiap kelas
print("\nMODEL PROBABILITY ANALYSIS")
print("-" * 55)

for class_id, name in enumerate(names):

    class_probs = probabilities[class_id]

    print(
        f"{name:<14}"
        f"Mean: {class_probs.mean():.4f}"
        f" | Max: {class_probs.max():.4f}"
    )

# Periksa probabilitas Background
# khusus pada area jaringan ground truth
tissue_region = (
    (gt_mask >= 1) &
    (gt_mask <= 5)
)

if np.any(tissue_region):

    print("\nBACKGROUND PROBABILITY ON TRUE TISSUE")

    print(
        "Mean:",
        round(
            probabilities[0][tissue_region].mean(),
            4
        )
    )


In [ ]:

print("SLIDING WINDOW MODEL COMPARISON")
print("-" * 55)

for name, path in [
    ("V5-A Full Resize", v5_a_path),
    ("V5-B Native Patch", v5_b_path)
]:

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    model.load_state_dict(
        ckpt["model_state"]
    )

    model.eval()

    pred, probs = sliding_window_predict(
        test_image,
        model,
        patch_size=320,
        stride=160
    )

    print(f"\n{name}")

    for class_id, class_name in enumerate(names):

        count = np.sum(pred == class_id)

        print(
            f"{class_name:<14}: "
            f"{count:>8,} pixels"
        )

    print(
        "Mean Background probability:",
        round(probs[0].mean(), 4)
    )


In [ ]:

# DataLoader untuk frozen training patches
frozen_loader = DataLoader(
    frozen_v5_samples,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

def diagnose_frozen_patches(checkpoint_path):

    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    model.load_state_dict(ckpt["model_state"])
    model.eval()

    confusion = torch.zeros(
        (6, 6),
        dtype=torch.int64
    )

    bg_probability_sum = 0.0
    tissue_pixels = 0

    with torch.inference_mode():

        for images, masks in frozen_loader:

            images = images.to(device)
            masks = masks.to(device)

            logits = model(
                normalize_images(images)
            )

            probs = torch.softmax(logits, dim=1)
            pred = logits.argmax(dim=1)

            valid = masks != 255
            tissue = (masks >= 1) & (masks <= 5)

            pairs = masks[valid] * 6 + pred[valid]

            confusion += torch.bincount(
                pairs,
                minlength=36
            ).reshape(6, 6).cpu()

            bg_probability_sum += (
                probs[:, 0][tissue].sum().item()
            )

            tissue_pixels += tissue.sum().item()

    tp = confusion.diag().float()
    actual = confusion.sum(dim=1).float()
    predicted = confusion.sum(dim=0).float()

    dice = 2 * tp / (actual + predicted).clamp_min(1)

    return {
        "dice": dice,
        "actual": actual,
        "predicted": predicted,
        "bg_on_tissue": bg_probability_sum / max(tissue_pixels, 1)
    }


for name, path in [
    ("V5-A Full Resize", v5_a_path),
    ("V5-B Native Patch", v5_b_path)
]:

    result = diagnose_frozen_patches(path)

    print("\n", name)
    print("-" * 45)

    print("Slough Dice:", round(result["dice"][2].item(), 4))
    print("Necrotic Dice:", round(result["dice"][4].item(), 4))

    print(
        "Actual tissue pixels:",
        int(result["actual"][1:6].sum().item())
    )

    print(
        "Predicted tissue pixels:",
        int(result["predicted"][1:6].sum().item())
    )

    print(
        "Mean Background probability on tissue:",
        round(result["bg_on_tissue"], 4)
    )


In [ ]:

# Load V5-B
ckpt = torch.load(
    v5_b_path,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(ckpt["model_state"])
model.eval()

# Inference padding lama
pred_old, probs_old = sliding_window_predict(
    test_image,
    model
)

# Buat padding simetris
h, w = test_image.shape[:2]

pad_h = max(0, 320 - h)
pad_w = max(0, 320 - w)

top = pad_h // 2
bottom = pad_h - top

left = pad_w // 2
right = pad_w - left

centered_image = np.pad(
    test_image,
    ((top, bottom), (left, right), (0, 0)),
    mode="edge"
)

# Inference pada gambar dengan padding simetris
pred_center, probs_center = sliding_window_predict(
    centered_image,
    model
)

# Kembalikan hasil ke ukuran foto asli
pred_center = pred_center[
    top:top + h,
    left:left + w
]

probs_center = probs_center[
    :,
    top:top + h,
    left:left + w
]

print("PADDING COMPARISON")
print("-" * 45)

print("Old predicted tissue:", np.sum(pred_old != 0))
print("Centered predicted tissue:", np.sum(pred_center != 0))

print(
    "Old mean BG probability:",
    round(probs_old[0].mean(), 4)
)

print(
    "Centered mean BG probability:",
    round(probs_center[0].mean(), 4)
)

print(
    "Centered BG probability on true tissue:",
    round(probs_center[0][tissue_region].mean(), 4)
)


In [ ]:

# Cari foto validation yang memiliki Slough
slough_cases = []

for entry in val_files:

    image_path, mask_path = entry.split()

    gt = np.array(
        Image.open(data_dir / mask_path)
    )

    slough_pixels = np.sum(gt == 2)

    if slough_pixels > 0:

        height, width = gt.shape

        slough_cases.append((
            height * width,
            image_path,
            mask_path,
            int(slough_pixels)
        ))

# Pilih tiga foto terkecil
diagnostic_cases = sorted(slough_cases)[:3]

print("VALIDATION DIAGNOSTIC CASES")
print("-" * 55)

for area, image_path, mask_path, slough in diagnostic_cases:

    print("\nImage:", image_path)
    print("Total pixels:", area)
    print("Slough pixels:", slough)


In [ ]:

print("NATIVE RESOLUTION VALIDATION TEST")
print("=" * 60)

for model_name, checkpoint_path in [
    ("V5-A Full Resize", v5_a_path),
    ("V5-B Native Patch", v5_b_path)
]:

    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    model.load_state_dict(ckpt["model_state"])
    model.eval()

    print("\nMODEL:", model_name)
    print("-" * 55)

    for _, image_path, mask_path, _ in diagnostic_cases:

        image = np.array(
            Image.open(
                data_dir / image_path
            ).convert("RGB")
        )

        gt = np.array(
            Image.open(data_dir / mask_path)
        )

        # Inference tanpa menggunakan ground truth
        pred, probs = sliding_window_predict(
            image,
            model,
            patch_size=320,
            stride=160
        )

        assert pred.shape == gt.shape

        # Hitung Dice kelas Slough dan Necrotic
        print("\nImage:", image_path.split("/")[-1])

        for class_id in [2, 4]:

            actual = gt == class_id
            predicted = pred == class_id

            tp = np.sum(actual & predicted)
            denominator = actual.sum() + predicted.sum()

            if denominator > 0:
                dice = 2 * tp / denominator
                print(
                    f"{names[class_id]} Dice: {dice:.4f}"
                )
            else:
                print(
                    f"{names[class_id]} Dice: N/A"
                )

        # Periksa kecenderungan Background
        print(
            "Predicted tissue:",
            int(np.sum(pred != 0))
        )

        tissue = (gt >= 1) & (gt <= 5)

        if np.any(tissue):
            print(
                "BG probability on true tissue:",
                round(probs[0][tissue].mean(), 4)
            )


In [ ]:

import torch.nn.functional as F

def predict_wound(image, model):

    # Ukuran asli foto
    original_h, original_w = image.shape[:2]

    # Resize sesuai preprocessing V5-A
    resized = Image.fromarray(image).resize(
        (320, 320),
        Image.Resampling.BILINEAR
    )

    # Konversi menjadi tensor BCHW
    tensor = torch.from_numpy(
        np.array(resized).copy()
    ).permute(2, 0, 1).float()

    tensor = tensor.unsqueeze(0).to(device) / 255.0

    # Normalisasi sesuai training
    tensor = normalize_images(tensor)

    model.eval()

    with torch.inference_mode():

        logits = model(tensor)

        # Ubah logits menjadi probabilitas
        probs = torch.softmax(logits, dim=1)

        # Kembalikan probabilitas ke resolusi asli
        probs = F.interpolate(
            probs,
            size=(original_h, original_w),
            mode="bilinear",
            align_corners=False
        )

        # Kelas dengan probabilitas terbesar
        prediction = probs.argmax(dim=1)[0]

    return (
        prediction.cpu().numpy().astype(np.uint8),
        probs[0].cpu().numpy()
    )

print("WoundVision inference function ready!")


In [ ]:

# Muat checkpoint V5-A
ckpt = torch.load(
    v5_a_path,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(ckpt["model_state"])
model.eval()

# Pilih foto validation
image_path = "Images/Patient_20/P20T_5.jpg"

image = np.array(
    Image.open(data_dir / image_path).convert("RGB")
)

# Prediksi tanpa menggunakan ground truth
pred, probs = predict_wound(image, model)

# Warna visualisasi setiap kelas
colors = np.array([
    [0, 0, 0],        # Background
    [0, 180, 255],    # Epithelial
    [255, 210, 0],    # Slough
    [230, 50, 70],    # Granulation
    [145, 60, 210],   # Necrotic
    [80, 200, 100]    # Other
], dtype=np.uint8)

# Konversi prediksi menjadi peta warna
color_mask = colors[pred]

# Buat overlay transparan
overlay = image.astype(np.float32).copy()

tissue = pred != 0

overlay[tissue] = (
    0.55 * overlay[tissue]
    + 0.45 * color_mask[tissue]
)

overlay = overlay.astype(np.uint8)

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(image)
axes[0].set_title("Original Wound Image")
axes[0].axis("off")

axes[1].imshow(overlay)
axes[1].set_title("WoundVision AI Prediction")
axes[1].axis("off")

plt.tight_layout()
plt.show()

# Ringkasan kelas yang diprediksi
print("WOUNDVISION AI INFERENCE")
print("-" * 45)

print("Image shape:", image.shape)
print("Prediction shape:", pred.shape)

predicted_tissue = np.sum(tissue)

print("\nPredicted tissue pixels:", predicted_tissue)

for class_id in range(1, 6):

    count = np.sum(pred == class_id)

    percentage = (
        count / predicted_tissue * 100
        if predicted_tissue > 0 else 0
    )

    print(
        f"{names[class_id]}: "
        f"{count:,} pixels ({percentage:.2f}%)"
    )

assert pred.shape == image.shape[:2]

print("\nWOUNDVISION INFERENCE: PASS")


In [ ]:

# Ground truth dari foto validation yang sama
gt = np.array(
    Image.open(
        data_dir / "Masks/Patient_20/P20T_5.png"
    )
)

assert gt.shape == pred.shape

# Fungsi overlay untuk label yang berbeda
def create_overlay(image, mask, colors):

    result = image.astype(np.float32).copy()

    # Hanya warnai kelas jaringan 1-5
    tissue = (mask >= 1) & (mask <= 5)

    result[tissue] = (
        0.55 * result[tissue]
        + 0.45 * colors[mask[tissue]]
    )

    return result.astype(np.uint8)


# Overlay ground truth dan prediction
gt_overlay = create_overlay(image, gt, colors)

pred_overlay = create_overlay(image, pred, colors)

# Tampilkan ketiganya
fig, axes = plt.subplots(
    1, 3, figsize=(16, 7)
)

axes[0].imshow(image)
axes[0].set_title("Original Image")

axes[1].imshow(gt_overlay)
axes[1].set_title("Ground Truth - LUTSeg")

axes[2].imshow(pred_overlay)
axes[2].set_title("WoundVision AI - Prediction")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

print("WOUNDVISION AI - IMAGE EVALUATION")
print("-" * 60)

valid = gt != 255

for class_id in range(1, 6):

    actual = (gt == class_id) & valid
    predicted = (pred == class_id) & valid

    tp = np.sum(actual & predicted)

    actual_count = np.sum(actual)
    predicted_count = np.sum(predicted)

    denominator = actual_count + predicted_count

    if denominator > 0:
        dice = 2 * tp / denominator
    else:
        dice = None

    print(f"\n{names[class_id]}")
    print("Ground truth:", actual_count)
    print("Predicted:", predicted_count)

    if dice is None:
        print("Dice: N/A")
    else:
        print(f"Dice: {dice:.4f}")


In [ ]:
%pip -q install gradio

In [ ]:

import gradio as gr

# Pastikan checkpoint MVP adalah V5-A
mvp_checkpoint = torch.load(
    v5_a_path,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(
    mvp_checkpoint["model_state"]
)

model.to(device)
model.eval()


def analyze_wound_upload(uploaded_image):

    if uploaded_image is None:
        raise gr.Error("Silakan upload foto terlebih dahulu.")

    # Gradio memberikan gambar dalam format RGB NumPy
    image_rgb = np.asarray(
        uploaded_image,
        dtype=np.uint8
    )

    # Gunakan fungsi inference yang sudah kita bangun
    pred, probs = predict_wound(
        image_rgb,
        model
    )

    # Buat overlay
    overlay = image_rgb.astype(np.float32).copy()
    colored_mask = colors[pred]

    tissue = (pred >= 1) & (pred <= 5)

    overlay[tissue] = (
        0.55 * overlay[tissue]
        + 0.45 * colored_mask[tissue]
    )

    overlay = overlay.astype(np.uint8)

    # Hitung persentase jaringan menurut model
    total_tissue = int(tissue.sum())

    rows = []

    for class_id in range(1, 6):

        count = int((pred == class_id).sum())

        percent = (
            count / total_tissue * 100
            if total_tissue > 0 else 0.0
        )

        rows.append([
            names[class_id],
            count,
            round(percent, 2)
        ])

    return overlay, rows


# Uji fungsi tanpa browser terlebih dahulu
test_overlay, test_rows = analyze_wound_upload(image)

print("GRADIO FUNCTION TEST")
print("-" * 45)
print("Overlay shape:", test_overlay.shape)
print("Number of tissue classes:", len(test_rows))
print("Results:", test_rows)

assert test_overlay.shape == image.shape
assert len(test_rows) == 5

print("\nGRADIO FUNCTION: PASS")


In [ ]:

print("FUNCTION CHECK")
print("-" * 40)

required = [
    "gr",
    "model",
    "predict_wound",
    "colors",
    "names",
    "analyze_wound_upload"
]

for name in required:
    print(f"{name}: {name in globals()}")


In [ ]:

demo = gr.Interface(
    fn=analyze_wound_upload,

    inputs=gr.Image(
        type="numpy",
        image_mode="RGB",
        sources=["upload"],
        label="Upload Foto Luka"
    ),

    outputs=[
        gr.Image(
            label="Hasil Segmentasi WoundVision AI"
        ),

        gr.Dataframe(
            headers=[
                "Jaringan",
                "Piksel",
                "Persentase (%)"
            ],
            datatype=[
                "str",
                "number",
                "number"
            ],
            interactive=False,
            label="Komposisi Jaringan (Prediksi)"
        )
    ],

    title="WoundVision AI",

    description=(
        "Research Prototype — Segmentasi jaringan luka "
        "menggunakan U-Net ResNet34. "
        "Hasil merupakan prediksi AI yang belum "
        "tervalidasi untuk penggunaan klinis. "
        "Bukan alat diagnosis atau dasar keputusan terapi."
    ),

    flagging_mode="never",
    analytics_enabled=False
)

# Jalankan untuk pengujian lokal
demo.launch(share=False)


In [ ]:

# Ambil checkpoint V5-A yang kita pilih untuk MVP
ckpt = torch.load(
    v5_a_path,
    map_location="cpu",
    weights_only=True
)

# Siapkan model khusus inference
inference_checkpoint = {
    "model_state": ckpt["model_state"],
    "architecture": "Unet",
    "encoder": "resnet34",
    "in_channels": 3,
    "classes": 6,
    "image_size": 320,
    "mean": [0.485, 0.456, 0.406],
    "std": [0.229, 0.224, 0.225],
    "class_names": names
}

# Simpan dengan nama unik
export_name = (
    "woundvision_mvp_"
    + uuid.uuid4().hex[:8]
    + ".pth"
)

export_path = save_checkpoint_safe(
    export_name,
    inference_checkpoint
)

print("MODEL EXPORT SUCCESS")
print("File:", export_path.name)
print("Size MB:", round(
    export_path.stat().st_size / (1024 ** 2), 2
))


In [ ]:

from pathlib import Path
import torch

# File yang sama dengan model di MacBook
mvp_path = Path(
    "/content/drive/MyDrive/WoundVisionAI/"
    "woundvision_mvp_3d6d1c55.pth"
)

assert mvp_path.exists(), "Checkpoint MVP tidak ditemukan"

# Evaluasi seluruh 30 gambar validation
results = evaluate_checkpoint(mvp_path)

cm = results["confusion"]

# Overall pixel accuracy
correct_pixels = cm.diag().sum().item()
total_pixels = cm.sum().item()

accuracy = correct_pixels / total_pixels

print("WOUNDVISION AI - FINAL MODEL EVALUATION")
print("=" * 55)

print("Checkpoint:", mvp_path.name)
print("Validation images:", len(val_files))

print(
    f"\nOverall Pixel Accuracy: {accuracy * 100:.2f}%"
)

print(
    f"Macro Tissue Dice: "
    f"{results['macro_dice'] * 100:.2f}%"
)

print("\nPER-CLASS METRICS")
print("-" * 55)

for i, name in enumerate(names):

    dice = results["dice"][i].item()
    iou = results["iou"][i].item()
    precision = results["precision"][i].item()
    recall = results["recall"][i].item()

    print(f"\n{name}")
    print(f"  Dice      : {dice * 100:.2f}%")
    print(f"  IoU       : {iou * 100:.2f}%")
    print(f"  Precision : {precision * 100:.2f}%")
    print(f"  Recall    : {recall * 100:.2f}%")


In [ ]:

# Confusion matrix evaluasi MVP
cm = results["confusion"].float()

# Seluruh kelas jaringan 1-5 dianggap area luka
# Background = kelas 0

# Prediksi jaringan pada lokasi jaringan sebenarnya
tp_wound = cm[1:6, 1:6].sum()

# Jaringan sebenarnya diprediksi Background
fn_wound = cm[1:6, 0].sum()

# Background diprediksi sebagai jaringan
fp_wound = cm[0, 1:6].sum()

# Dice segmentasi area luka
wound_dice = (
    2 * tp_wound
    / (2 * tp_wound + fp_wound + fn_wound)
)

# Akurasi identifikasi jenis jaringan
# pada seluruh piksel yang sebenarnya jaringan
correct_tissue_class = cm.diag()[1:6].sum()
total_true_tissue = cm[1:6, :].sum()

tissue_class_accuracy = (
    correct_tissue_class / total_true_tissue
)

print("WOUNDVISION AI - STAGE ANALYSIS")
print("-" * 50)

print(
    "Wound Segmentation Dice:",
    f"{wound_dice.item() * 100:.2f}%"
)

print(
    "Tissue Classification Accuracy:",
    f"{tissue_class_accuracy.item() * 100:.2f}%"
)

print(
    "Macro Tissue Dice:",
    f"{results['macro_dice'] * 100:.2f}%"
)


In [ ]:

from pathlib import Path
import hashlib

# Checkpoint yang sama dengan aplikasi lokal
mvp_path = Path(
    "/content/drive/MyDrive/WoundVisionAI/"
    "woundvision_mvp_3d6d1c55.pth"
)

assert mvp_path.exists(), "Checkpoint tidak ditemukan"

# Evaluasi ulang dengan variabel terpisah
mvp_results = evaluate_checkpoint(mvp_path)

cm = mvp_results["confusion"].detach().cpu().float()

# Hitung ulang tiga metrik
tp = cm[1:6, 1:6].sum()
fn = cm[1:6, 0].sum()
fp = cm[0, 1:6].sum()

wound_dice = 2 * tp / (2 * tp + fn + fp)

correct_tissue = cm.diag()[1:6].sum()
total_tissue = cm[1:6, :].sum()

tissue_accuracy = correct_tissue / total_tissue

# Fingerprint confusion matrix
fingerprint = hashlib.sha256(
    cm.numpy().tobytes()
).hexdigest()[:12]

print("CHECKPOINT VERIFICATION")
print("=" * 50)
print("Checkpoint:", mvp_path.name)
print("Validation images:", len(val_loader.dataset))
print("Valid pixels:", int(cm.sum()))
print("Confusion fingerprint:", fingerprint)

print("\nMETRICS")
print("-" * 50)
print(f"Pixel Accuracy: {cm.diag().sum() / cm.sum() * 100:.2f}%")
print(f"Wound Dice: {wound_dice * 100:.2f}%")
print(f"Tissue Accuracy: {tissue_accuracy * 100:.2f}%")
print(
    f"Macro Tissue Dice: "
    f"{float(mvp_results['macro_dice']) * 100:.2f}%"
)


In [ ]:

# Gunakan hasil checkpoint yang sudah diverifikasi
cm = mvp_results["confusion"].detach().cpu().float()

# Total piksel dengan ground truth jaringan (kelas 1-5)
total_tissue = cm[1:6, :].sum()

# Kesalahan 1: Jaringan dianggap Background
missed = cm[1:6, 0].sum()

# Kesalahan 2: Jaringan terdeteksi,
# tetapi salah jenis
detected = cm[1:6, 1:6].sum()
correct = cm.diag()[1:6].sum()
wrong_class = detected - correct

# Prediksi yang benar
correct_percent = correct / total_tissue * 100

print("WOUNDVISION AI — ERROR ANALYSIS")
print("=" * 55)

print(
    f"Missed as Background : "
    f"{missed / total_tissue * 100:.2f}%"
)

print(
    f"Wrong Tissue Class   : "
    f"{wrong_class / total_tissue * 100:.2f}%"
)

print(
    f"Correct Tissue Class : "
    f"{correct_percent:.2f}%"
)

print("\nTISSUE CLASSIFICATION WHEN DETECTED")
print("-" * 55)

conditional_accuracy = correct / detected.clamp_min(1)

print(
    f"Conditional Accuracy : "
    f"{conditional_accuracy * 100:.2f}%"
)

assert torch.allclose(
    missed + wrong_class + correct,
    total_tissue
)

print("\nERROR ANALYSIS: PASS")


In [ ]:

cm = mvp_results["confusion"].detach().cpu().long()

class_names = [
    "Background", "Epithelial", "Slough",
    "Granulation", "Necrotic", "Other"
]

errors = []

# Baris = label asli, kolom = prediksi
for true_id in range(1, 6):
    for pred_id in range(1, 6):

        if true_id == pred_id:
            continue

        count = int(cm[true_id, pred_id])
        class_total = int(cm[true_id, :].sum())

        if count > 0:
            errors.append((
                class_names[true_id],
                class_names[pred_id],
                count,
                100 * count / class_total
            ))

errors.sort(key=lambda x: x[2], reverse=True)

print("TOP TISSUE CONFUSIONS")
print("=" * 65)

for actual, predicted, count, percent in errors[:10]:
    print(
        f"{actual:13} -> {predicted:13} "
        f"{count:>8,} px ({percent:.2f}% of true class)"
    )


In [ ]:

from pathlib import Path
import torch

# Checkpoint MVP yang sudah berhasil diverifikasi
recovered_path = Path(
    "/content/woundvision_mvp_3d6d1c55.pth"
)

print("WOUNDVISION V0.2 - PRECHECK")
print("-" * 45)

print("Checkpoint exists:", recovered_path.exists())
print("Train dataset:", "train_dataset" in globals())
print("Validation loader:", "val_loader" in globals())

assert recovered_path.exists(), (
    "Checkpoint belum ada di /content/. "
    "Upload ulang file .pth dari MacBook."
)

assert "train_dataset" in globals(), (
    "Jalankan ulang cell persiapan train_dataset."
)

assert "val_loader" in globals(), (
    "Jalankan ulang cell persiapan val_loader."
)

# Pastikan checkpoint benar-benar bisa dibaca
checkpoint = torch.load(
    recovered_path,
    map_location="cpu",
    weights_only=True
)

assert "model_state" in checkpoint

print("\nCHECKPOINT: PASS")
print("DATASET SETUP: PASS")


In [ ]:

import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

from torch.utils.data import DataLoader

torch.manual_seed(2026)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Model baru, bobot awal dari MVP
v02_model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=6,
    activation=None
).to(device)

checkpoint = torch.load(
    recovered_path,
    map_location="cpu",
    weights_only=True
)

v02_model.load_state_dict(checkpoint["model_state"])

# Training loader
v02_train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(2026)
)

# Loss sama seperti baseline V5-A
class_weights = torch.tensor(
    [0.25, 1.0, 1.3412, 1.0135, 3.0, 1.0],
    device=device
)

ce = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=255
)

dice_loss = smp.losses.DiceLoss(
    mode="multiclass",
    from_logits=True,
    ignore_index=255
)

optimizer_v02 = torch.optim.AdamW(
    v02_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

mean = torch.tensor(
    [0.485, 0.456, 0.406],
    device=device
).view(1, 3, 1, 1)

std = torch.tensor(
    [0.229, 0.224, 0.225],
    device=device
).view(1, 3, 1, 1)

def preprocess(images):
    return (images.to(device) - mean) / std

print("V0.2 TRAINING SETUP: PASS")
print("Device:", device)
print("Train images:", len(train_dataset))
print("Validation images:", len(val_loader.dataset))



In [ ]:

from pathlib import Path
import uuid

def validate_v02(model):
    model.eval()
    cm = torch.zeros((6, 6), dtype=torch.long)

    with torch.inference_mode():
        for images, masks in val_loader:
            images = preprocess(images)
            masks = masks.to(device)

            logits = model(images)
            preds = logits.argmax(dim=1)

            valid = masks != 255

            indices = (
                masks[valid] * 6 + preds[valid]
            )

            cm += torch.bincount(
                indices.cpu(),
                minlength=36
            ).reshape(6, 6)

    tp = cm.diag().float()
    total_true = cm.sum(dim=1).float()
    total_pred = cm.sum(dim=0).float()

    dice = 2 * tp / (
        total_true + total_pred
    ).clamp_min(1)

    macro_tissue_dice = dice[1:6].mean().item()

    wound_tp = cm[1:6, 1:6].sum().float()
    wound_fn = cm[1:6, 0].sum().float()
    wound_fp = cm[0, 1:6].sum().float()

    wound_dice = (
        2 * wound_tp /
        (2 * wound_tp + wound_fn + wound_fp)
    ).item()

    return macro_tissue_dice, wound_dice, dice


# Pastikan evaluasi sama dengan baseline
baseline_macro, baseline_wound, _ = validate_v02(
    v02_model
)

print(f"Baseline Macro Dice: {baseline_macro:.4f}")
print(f"Baseline Wound Dice: {baseline_wound:.4f}")

assert abs(baseline_macro - 0.2418) < 0.005, (
    "Baseline tidak cocok. Periksa preprocessing."
)

best_macro = baseline_macro
best_epoch = 0

save_path = Path("/content") / (
    f"woundvision_v02_{uuid.uuid4().hex[:8]}.pth"
)

for epoch in range(1, 4):
    v02_model.train()
    total_loss = 0

    for images, masks in v02_train_loader:
        images = preprocess(images)
        masks = masks.to(device)

        optimizer_v02.zero_grad()

        logits = v02_model(images)

        loss = (
            ce(logits, masks)
            + dice_loss(logits, masks)
        )

        loss.backward()
        optimizer_v02.step()

        total_loss += loss.item()

    macro, wound, per_class = validate_v02(
        v02_model
    )

    print(
        f"Epoch {epoch} | "
        f"Loss: {total_loss / len(v02_train_loader):.4f} | "
        f"Macro Dice: {macro*100:.2f}% | "
        f"Wound Dice: {wound*100:.2f}%"
    )

    if macro > best_macro:
        best_macro = macro
        best_epoch = epoch

        torch.save(
            {
                "model_state": v02_model.state_dict(),
                "epoch": epoch,
                "macro_tissue_dice": macro,
                "architecture": "Unet",
                "encoder": "resnet34",
                "classes": 6,
                "image_size": 320
            },
            save_path
        )

print("\nEXPERIMENT COMPLETE")
print("Best epoch:", best_epoch)
print(f"Best Macro Tissue Dice: {best_macro*100:.2f}%")

if best_epoch > 0:
    print("Improved checkpoint:", save_path)
else:
    print("No improvement over v0.1 baseline")


In [ ]:

from google.colab import files
from pathlib import Path
import torch
import hashlib
import zipfile

best_path = Path(
    "/content/woundvision_v02_78189535.pth"
)

assert best_path.exists()

assert zipfile.is_zipfile(best_path), (
    "Checkpoint tidak valid"
)

ckpt = torch.load(
    best_path,
    map_location="cpu",
    weights_only=True
)

assert "model_state" in ckpt

sha256 = hashlib.sha256(
    best_path.read_bytes()
).hexdigest()

print("CHECKPOINT V0.2 VERIFIED")
print("File:", best_path.name)
print("SHA256:", sha256)
print("Size MB:", round(best_path.stat().st_size / 1024**2, 2))

# Download ke MacBook
files.download(str(best_path))


In [ ]:

# Load bobot checkpoint terbaik
v02_model.load_state_dict(
    ckpt["model_state"]
)

v02_model.eval()

# Evaluasi ulang v0.2
macro_v02, wound_v02, dice_v02 = validate_v02(
    v02_model
)

# Hitung Dice v0.1 dari confusion matrix
cm = mvp_results["confusion"].float()

tp = cm.diag()
total_true = cm.sum(dim=1)
total_pred = cm.sum(dim=0)

dice_v01 = (
    2 * tp / (total_true + total_pred).clamp_min(1)
)

class_names = [
    "Background",
    "Epithelial",
    "Slough",
    "Granulation",
    "Necrotic",
    "Other"
]

print("V0.1 VS V0.2 - PER CLASS")
print("=" * 60)

for i in range(1, 6):
    old = dice_v01[i].item() * 100
    new = dice_v02[i].item() * 100
    change = new - old

    print(
        f"{class_names[i]:13} | "
        f"V0.1: {old:6.2f}% | "
        f"V0.2: {new:6.2f}% | "
        f"Delta: {change:+6.2f}"
    )

print("\nMacro Tissue Dice:", f"{macro_v02*100:.2f}%")
print("Wound Dice:", f"{wound_v02*100:.2f}%")


In [ ]:

import torch
from pathlib import Path

# Pastikan menggunakan checkpoint V0.2 terbaik
v02_path = Path(
    "/content/woundvision_v02_78189535.pth"
)

assert v02_path.exists(), (
    "Checkpoint V0.2 tidak ditemukan. "
    "Upload ulang file yang sudah diunduh."
)

v02_ckpt = torch.load(
    v02_path,
    map_location="cpu",
    weights_only=True
)

v02_model.load_state_dict(v02_ckpt["model_state"])
v02_model.eval()

# Confusion matrix validation V0.2
cm_v02 = torch.zeros(
    (6, 6),
    dtype=torch.long
)

with torch.inference_mode():
    for images, masks in val_loader:

        images = preprocess(images)
        masks = masks.to(device)

        logits = v02_model(images)
        preds = logits.argmax(dim=1)

        valid = masks != 255

        indices = (
            masks[valid].long() * 6
            + preds[valid].long()
        )

        cm_v02 += torch.bincount(
            indices.cpu(),
            minlength=36
        ).reshape(6, 6)

names = [
    "Background",
    "Epithelial",
    "Slough",
    "Granulation",
    "Necrotic",
    "Other"
]

print("NECROTIC PREDICTION AUDIT — V0.2")
print("=" * 55)

# Baris 4 = seluruh piksel dengan label asli Necrotic
necrotic_gt = cm_v02[4]
total = int(necrotic_gt.sum())

print("Total ground-truth Necrotic:", f"{total:,}")
print()

for class_id, name in enumerate(names):

    count = int(necrotic_gt[class_id])
    percentage = count / total * 100 if total else 0

    print(
        f"Necrotic -> {name:13}: "
        f"{count:>8,} px ({percentage:6.2f}%)"
    )

print("\nNECROTIC AUDIT: PASS")


In [ ]:

import torch
from pathlib import Path

# Pastikan menggunakan checkpoint V0.2 terbaik
v02_path = Path(
    "/content/woundvision_v02_78189535.pth"
)

assert v02_path.exists(), (
    "Checkpoint V0.2 tidak ditemukan. "
    "Upload ulang file yang sudah diunduh."
)

v02_ckpt = torch.load(
    v02_path,
    map_location="cpu",
    weights_only=True
)

v02_model.load_state_dict(v02_ckpt["model_state"])
v02_model.eval()

# Confusion matrix validation V0.2
cm_v02 = torch.zeros(
    (6, 6),
    dtype=torch.long
)

with torch.inference_mode():
    for images, masks in val_loader:

        images = preprocess(images)
        masks = masks.to(device)

        logits = v02_model(images)
        preds = logits.argmax(dim=1)

        valid = masks != 255

        indices = (
            masks[valid].long() * 6
            + preds[valid].long()
        )

        cm_v02 += torch.bincount(
            indices.cpu(),
            minlength=36
        ).reshape(6, 6)

names = [
    "Background",
    "Epithelial",
    "Slough",
    "Granulation",
    "Necrotic",
    "Other"
]

print("NECROTIC PREDICTION AUDIT — V0.2")
print("=" * 55)

# Baris 4 = seluruh piksel dengan label asli Necrotic
necrotic_gt = cm_v02[4]
total = int(necrotic_gt.sum())

print("Total ground-truth Necrotic:", f"{total:,}")
print()

for class_id, name in enumerate(names):

    count = int(necrotic_gt[class_id])
    percentage = count / total * 100 if total else 0

    print(
        f"Necrotic -> {name:13}: "
        f"{count:>8,} px ({percentage:6.2f}%)"
    )

print("\nNECROTIC AUDIT: PASS")
